In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
"""
Dataset Diagnostic Audit Script  (Single-Dataset Version)
===========================================================
Answers three questions before any model code is written:

  Q1. Epoch length (seconds) in the .fif files
  Q2. Sampling rate of the .fif epoch files
  Q3. Per-subject epoch counts broken down by class label

The script auto-detects the dataset type from filenames:
  • Adora Epilepsy  → labels: ictal / interictal
  • MAAED Emotion   → labels: positive / negative

Dependencies:
    pip install mne pandas

Usage:
    1. Set DATASET_DIR below to your folder path.
    2. python dataset_audit.py

Output:
    • Printed report in terminal
    • dataset_audit_results.csv saved in the current directory

Author: Research Audit Script
Date:   2026-03-12
"""

from __future__ import annotations

import re
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import mne
import pandas as pd

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
#  ▶  SET THIS ONE PATH BEFORE RUNNING
#     Examples:
#       DATASET_DIR = Path(r"C:/data/adora_epilepsy_eeg/Final_cleaned")
#       DATASET_DIR = Path(r"C:/data/MAAED_Cleaned_Epochs_Archive_new")
# ─────────────────────────────────────────────────────────────────────────────
DATASET_DIR = Path(r"/kaggle/input/datasets/johanliebert28/maaed-cleaned-epochs-archive-new")
# ─────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — Data Structures
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class EpochMeta:
    """
    Metadata extracted from a single .fif epochs file.

    Attributes
    ----------
    filepath      : absolute path to the file
    subject_id    : parsed subject identifier string
    label         : class label (ictal/interictal/positive/negative/unknown)
    n_epochs      : number of epochs that survived after artifact rejection
    n_channels    : number of EEG channels
    sfreq         : sampling frequency in Hz
    tmin          : epoch start time in seconds relative to event
    tmax          : epoch end time in seconds relative to event
    epoch_len_sec : tmax - tmin (actual epoch duration)
    ch_names      : list of channel name strings
    error         : non-None string if the file failed to load
    """
    filepath:      str
    subject_id:    str
    label:         str
    n_epochs:      int
    n_channels:    int
    sfreq:         float
    tmin:          float
    tmax:          float
    epoch_len_sec: float
    ch_names:      list[str]     = field(default_factory=list)
    error:         Optional[str] = None


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — Filename Parsers
# ─────────────────────────────────────────────────────────────────────────────

def _detect_dataset_type(fif_files: list[Path]) -> str:
    """
    Inspect the first few filenames and return 'adora' or 'maaed'.

    MAAED files follow: {gender}_{valence}_raw_{id}_clean-epo.fif
    Adora files follow: arbitrary clinical names (no gender prefix pattern)

    Returns
    -------
    'maaed'  if MAAED naming pattern is detected
    'adora'  otherwise (default)
    """
    maaed_pattern = re.compile(
        r"^(male|female)_(positive|negative)_raw_\d+_clean-epo\.fif$",
        re.IGNORECASE
    )
    sample = [f.name for f in fif_files[:20]]
    n_maaed = sum(1 for n in sample if maaed_pattern.match(n))
    return "maaed" if n_maaed >= max(1, len(sample) // 2) else "adora"


def _parse_maaed(filename: str) -> tuple[str, str]:
    """
    Parse MAAED filename: {gender}_{valence}_raw_{id}_clean-epo.fif

    Returns
    -------
    (subject_id, label)
        subject_id : e.g. 'female_12'
        label      : 'positive' | 'negative' | 'unknown'
    """
    name  = filename.lower()
    match = re.match(
        r"^(male|female)_(positive|negative)_raw_(\d+)_",
        name
    )
    if match:
        gender, valence, sid = match.group(1), match.group(2), match.group(3)
        return f"{gender}_{sid}", valence
    return "unknown", "unknown"


def _parse_adora(filename: str) -> tuple[str, str]:
    """
    Parse Adora Epilepsy filename.

    Label inference (case-insensitive keyword matching):
        ictal / seizure / epil  (but NOT 'inter') → 'ictal'
        inter / normal / non                       → 'interictal'
        everything else                            → 'unknown'

    Subject ID: first integer found in the filename.

    Returns
    -------
    (subject_id, label)
    """
    name = filename.lower()

    # Label
    if any(k in name for k in ["seizure", "epil"]):
        label = "ictal"
    elif "ictal" in name and "inter" not in name:
        label = "ictal"
    elif any(k in name for k in ["inter", "normal", "non"]):
        label = "interictal"
    else:
        label = "unknown"

    # Subject ID
    sid_match = re.search(r"(\d+)", name)
    subject_id = sid_match.group(1) if sid_match else "unknown"

    return subject_id, label


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — File Scanner
# ─────────────────────────────────────────────────────────────────────────────

def _safe_read(fif_path: Path) -> tuple[Optional[mne.Epochs], Optional[str]]:
    """
    Attempt to read a .fif epochs file without crashing the script.

    Returns
    -------
    (epochs, None)   on success
    (None, error_msg) on failure
    """
    try:
        epochs = mne.read_epochs(str(fif_path), preload=False, verbose=False)
        return epochs, None
    except Exception as exc:
        return None, str(exc)


def scan_dataset(directory: Path) -> tuple[list[EpochMeta], str]:
    """
    Scan all .fif files in *directory* (recursive), extract metadata per file.

    Parameters
    ----------
    directory : Path
        Root folder to scan recursively.

    Returns
    -------
    records      : list of EpochMeta (one per file)
    dataset_type : 'adora' | 'maaed'
    """
    # Collect .fif files; glob patterns cover both naming conventions
    fif_files = sorted(set(
        list(directory.glob("**/*-epo.fif")) +
        list(directory.glob("**/*_epo.fif")) +
        list(directory.glob("**/*.fif"))
    ))

    if not fif_files:
        print(f"\n  [WARNING] No .fif files found in:\n    {directory}")
        print("  Check DATASET_DIR is pointing to the correct folder.\n")
        return [], "unknown"

    dataset_type = _detect_dataset_type(fif_files)
    parse_fn     = _parse_maaed if dataset_type == "maaed" else _parse_adora

    print(f"\n  Detected dataset type : {dataset_type.upper()}")
    print(f"  Files found           : {len(fif_files)}")
    print(f"  Directory             : {directory}\n")

    records: list[EpochMeta] = []

    for fp in fif_files:
        subject_id, label = parse_fn(fp.name)
        epochs, err       = _safe_read(fp)

        if err or epochs is None:
            records.append(EpochMeta(
                filepath=str(fp), subject_id=subject_id, label=label,
                n_epochs=0, n_channels=0, sfreq=0.0,
                tmin=0.0, tmax=0.0, epoch_len_sec=0.0,
                ch_names=[], error=err,
            ))
            continue

        info      = epochs.info
        sfreq     = float(info["sfreq"])
        tmin      = float(epochs.tmin)
        tmax      = float(epochs.tmax)
        epoch_len = round(tmax - tmin, 6)

        records.append(EpochMeta(
            filepath=str(fp),
            subject_id=subject_id,
            label=label,
            n_epochs=len(epochs),
            n_channels=len(info["ch_names"]),
            sfreq=sfreq,
            tmin=tmin,
            tmax=tmax,
            epoch_len_sec=epoch_len,
            ch_names=info["ch_names"],
        ))

    return records, dataset_type


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — Report Printer
# ─────────────────────────────────────────────────────────────────────────────

def _line(char: str = "═", width: int = 68) -> None:
    print(char * width)


def print_report(records: list[EpochMeta], dataset_type: str) -> None:
    """
    Print the three-question diagnostic report to stdout.

    Parameters
    ----------
    records      : output of scan_dataset()
    dataset_type : 'adora' | 'maaed'
    """
    _line()
    label_map = {"adora": "ADORA EPILEPSY EEG", "maaed": "MAAED EMOTION EEG"}
    print(f"  {label_map.get(dataset_type, 'UNKNOWN DATASET')} — DIAGNOSTIC REPORT")
    _line()

    if not records:
        print("  No records loaded. Cannot generate report.")
        return

    ok     = [r for r in records if r.error is None]
    failed = [r for r in records if r.error is not None]

    print(f"\n  Total files scanned : {len(records)}")
    print(f"  Successfully loaded : {len(ok)}")
    print(f"  Failed to load      : {len(failed)}")

    if not ok:
        print("\n  All files failed. Check file integrity and MNE version.")
        return

    # ── Q1 / Q2 : Epoch length and Sampling Rate ─────────────────────────
    _line("─")
    print("  Q1 + Q2 ▶  EPOCH LENGTH (s) and SAMPLING RATE (Hz)")
    _line("─")

    lengths = sorted(set(r.epoch_len_sec for r in ok))
    sfreqs  = sorted(set(r.sfreq         for r in ok))

    print(f"\n  Epoch length(s) found:")
    for L in lengths:
        n = sum(1 for r in ok if r.epoch_len_sec == L)
        print(f"    {L:.4f} s  ←  {n} file(s)")

    if len(lengths) == 1:
        print(f"\n  ✔  Uniform epoch length : {lengths[0]:.4f} s")
    else:
        print(f"\n  ✘  INCONSISTENT epoch lengths — investigate before training.")

    print(f"\n  Sampling rate(s) found:")
    for sf in sfreqs:
        n = sum(1 for r in ok if r.sfreq == sf)
        print(f"    {sf:.1f} Hz  ←  {n} file(s)")

    if len(sfreqs) == 1:
        print(f"\n  ✔  Uniform sampling rate : {sfreqs[0]:.1f} Hz")
    else:
        print(f"\n  ✘  INCONSISTENT sampling rates — resample to a common rate.")

    # ── Channel info ─────────────────────────────────────────────────────
    _line("─")
    print("  Q1 supplement ▶  CHANNEL COUNT")
    _line("─")
    ch_counts = sorted(set(r.n_channels for r in ok))
    for nc in ch_counts:
        n       = sum(1 for r in ok if r.n_channels == nc)
        example = next(r for r in ok if r.n_channels == nc)
        print(f"\n  {nc} channels  ← {n} file(s)")
        print(f"  First 8 channel names: {example.ch_names[:8]}")

    # ── Q3 : Per-subject epoch counts ────────────────────────────────────
    _line("─")
    print("  Q3 ▶  PER-SUBJECT EPOCH COUNTS BY CLASS LABEL")
    _line("─")

    df = pd.DataFrame([{
        "subject_id":  r.subject_id,
        "label":       r.label,
        "n_epochs":    r.n_epochs,
        "sfreq":       r.sfreq,
        "epoch_len_s": r.epoch_len_sec,
        "n_channels":  r.n_channels,
    } for r in ok])

    # Pivot: rows = subjects, columns = labels
    pivot = df.pivot_table(
        index="subject_id",
        columns="label",
        values="n_epochs",
        aggfunc="sum",
        fill_value=0,
    )

    print()
    print(pivot.to_string())
    print()

    # Summary statistics per label
    _line("─")
    print("  SUMMARY STATISTICS PER LABEL")
    _line("─")
    for lbl in pivot.columns:
        total = pivot[lbl].sum()
        mn    = pivot[lbl].min()
        mx    = pivot[lbl].max()
        med   = pivot[lbl].median()
        print(f"\n  [{lbl}]")
        print(f"    Total epochs  : {total}")
        print(f"    Min / subject : {mn}")
        print(f"    Max / subject : {mx}")
        print(f"    Median        : {med:.0f}")

    # Class imbalance — only relevant for Adora
    if dataset_type == "adora":
        _line("─")
        print("  CLASS IMBALANCE CHECK (Adora)")
        _line("─")
        if "ictal" in pivot.columns and "interictal" in pivot.columns:
            n_ictal      = pivot["ictal"].sum()
            n_interictal = pivot["interictal"].sum()
            ratio        = n_interictal / max(n_ictal, 1)
            print(f"\n  Ictal epochs      : {n_ictal}")
            print(f"  Interictal epochs : {n_interictal}")
            print(f"  Imbalance ratio   : {ratio:.2f} : 1  (interictal : ictal)")
            if ratio > 5:
                print("\n  ✔  Severe imbalance confirmed.")
                print("     VAE anomaly detection (Direction B) is justified.")
                print("     Do NOT use accuracy as the primary metric.")
                print("     Required metrics: AUROC, Sensitivity @ 95% Specificity, F1.")
            else:
                print("\n  ⚠  Mild imbalance. Verify label assignments from filenames.")
        elif "unknown" in pivot.columns and pivot.columns.tolist() == ["unknown"]:
            print("\n  ✘  ALL labels are 'unknown'.")
            print("     The filename parser could not detect ictal/interictal keywords.")
            print("     Post 5 sample filenames from your dataset so the parser can be fixed.")

    # Files that failed to load
    if failed:
        _line("─")
        print(f"  FAILED FILES ({len(failed)} total)")
        _line("─")
        for r in failed:
            print(f"  {Path(r.filepath).name}")
            print(f"    Error: {r.error}")

    _line()


# ─���───────────────────────────────────────────────────────────────────────────
# SECTION 5 — Main Entrypoint
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    """Entry point: validate path, scan, report, export CSV."""
    _line("═")
    print("  DATASET AUDIT — PRE-TRAINING DIAGNOSTIC")
    print("  Run this BEFORE any model code is written")
    _line("═")

    # Path validation
    if not DATASET_DIR.exists():
        print(f"\n  ✘  DATASET_DIR not found:\n    {DATASET_DIR}")
        print("\n  Fix the DATASET_DIR variable at the top of this script.")
        return

    # Scan
    records, dataset_type = scan_dataset(DATASET_DIR)

    # Print report
    print_report(records, dataset_type)

    # Export CSV
    if records:
        rows = [{
            "filepath":      r.filepath,
            "subject_id":    r.subject_id,
            "label":         r.label,
            "n_epochs":      r.n_epochs,
            "n_channels":    r.n_channels,
            "sfreq":         r.sfreq,
            "tmin":          r.tmin,
            "tmax":          r.tmax,
            "epoch_len_sec": r.epoch_len_sec,
            "ch_names":      "|".join(r.ch_names),
            "error":         r.error or "",
            "dataset_type":  dataset_type,
        } for r in records]

        out_csv = Path("dataset_audit_results.csv")
        pd.DataFrame(rows).to_csv(out_csv, index=False)
        print(f"\n  ✔  Full audit exported to: {out_csv.resolve()}\n")


if __name__ == "__main__":
    main()

════════════════════════════════════════════════════════════════════
  DATASET AUDIT — PRE-TRAINING DIAGNOSTIC
  Run this BEFORE any model code is written
════════════════════════════════════════════════════════════════════

  Detected dataset type : MAAED
  Files found           : 212
  Directory             : /kaggle/input/datasets/johanliebert28/maaed-cleaned-epochs-archive-new

════════════════════════════════════════════════════════════════════
  MAAED EMOTION EEG — DIAGNOSTIC REPORT
════════════════════════════════════════════════════════════════════

  Total files scanned : 212
  Successfully loaded : 212
  Failed to load      : 0
────────────────────────────────────────────────────────────────────
  Q1 + Q2 ▶  EPOCH LENGTH (s) and SAMPLING RATE (Hz)
────────────────────────────────────────────────────────────────────

  Epoch length(s) found:
    3.9922 s  ←  212 file(s)

  ✔  Uniform epoch length : 3.9922 s

  Sampling rate(s) found:
    128.0 Hz  ←  212 file(s)

  ✔  Uniform 

In [3]:
"""
Dataset Diagnostic Audit v2 — MAAED (Mount Adora Epilepsy EEG)
================================================================
Answers:
  Q1. Epoch length (seconds)
  Q2. Channel count after preprocessing (19 vs 20 issue)
  Q3. Exact channel names — identify the extra channel in 20-ch subjects
  Q4. Epileptic : Normal epoch ratio per subject
  Q5. Subjects with 0 epochs in one class — full breakdown
  Q6. Per-subject channel count table

Dependencies:
    pip install mne pandas

Usage:
    Set DATASET_DIR below, then: python dataset_audit_v2.py

Date: 2026-03-12
"""

from __future__ import annotations

import re
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import mne
import pandas as pd

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
#  ▶  SET THIS PATH
# ─────────────────────────────────────────────────────────────────────────────
DATASET_DIR = Path(r"/kaggle/input/datasets/johanliebert28/maaed-cleaned-epochs-archive-new")
# ───────────────────────────────────────────────────────────────────────────��─


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — Data Structure
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class EpochMeta:
    """
    Metadata for one .fif file.

    filepath      : absolute path
    subject_id    : e.g. 'female_12' or 'male_7'
    gender        : 'female' | 'male'
    label         : 'epileptic' | 'normal' | 'unknown'
      (positive → epileptic, negative → normal — confirm from output)
    raw_label     : original string from filename ('positive'/'negative')
    n_epochs      : surviving epochs after artifact rejection
    n_channels    : channel count
    sfreq         : sampling frequency Hz
    tmin          : epoch start (s)
    tmax          : epoch end (s)
    epoch_len_sec : tmax - tmin
    ch_names      : full list of channel names
    error         : load error string or None
    """
    filepath:      str
    subject_id:    str
    gender:        str
    label:         str
    raw_label:     str
    n_epochs:      int
    n_channels:    int
    sfreq:         float
    tmin:          float
    tmax:          float
    epoch_len_sec: float
    ch_names:      list[str]     = field(default_factory=list)
    error:         Optional[str] = None


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — Filename Parser
# ─────────────────────────────────────────────────────────────────────────────

def _parse_filename(filename: str) -> tuple[str, str, str, str]:
    """
    Parse MAAED filename pattern:
        {gender}_{valence}_raw_{id}_clean-epo.fif

    Maps:
        positive → 'epileptic'  (assumption — verify from output)
        negative → 'normal'     (assumption — verify from output)

    Returns
    -------
    (subject_id, gender, label, raw_label)
    Example: 'female_12', 'female', 'epileptic', 'positive'
    """
    name  = filename.lower()
    match = re.match(
        r"^(male|female)_(positive|negative)_raw_(\d+)_",
        name
    )
    if not match:
        return "unknown", "unknown", "unknown", "unknown"

    gender    = match.group(1)          # 'male' | 'female'
    raw_label = match.group(2)          # 'positive' | 'negative'
    sid_num   = match.group(3)          # '12'
    subject_id = f"{gender}_{sid_num}"  # 'female_12'

    # Map to clinical label
    # ⚠ ASSUMPTION: positive=epileptic, negative=normal
    # The audit output will confirm or deny this
    label = "epileptic" if raw_label == "positive" else "normal"

    return subject_id, gender, label, raw_label


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — File Reader
# ─────────────────────────────────────────────────────────────────────────────

def _safe_read(fif_path: Path) -> tuple[Optional[mne.Epochs], Optional[str]]:
    """Safe wrapper around mne.read_epochs. Never raises."""
    try:
        epochs = mne.read_epochs(str(fif_path), preload=False, verbose=False)
        return epochs, None
    except Exception as exc:
        return None, str(exc)


def scan_dataset(directory: Path) -> list[EpochMeta]:
    """
    Scan all .fif files recursively. Return one EpochMeta per file.

    Parameters
    ----------
    directory : root folder to scan

    Returns
    -------
    list[EpochMeta]
    """
    fif_files = sorted(set(
        list(directory.glob("**/*-epo.fif")) +
        list(directory.glob("**/*_epo.fif")) +
        list(directory.glob("**/*.fif"))
    ))

    if not fif_files:
        print(f"\n  [ERROR] No .fif files found in:\n    {directory}")
        return []

    print(f"\n  Files found : {len(fif_files)}")
    print(f"  Scanning...  (this may take 1-2 minutes)\n")

    records: list[EpochMeta] = []

    for fp in fif_files:
        subject_id, gender, label, raw_label = _parse_filename(fp.name)
        epochs, err = _safe_read(fp)

        if err or epochs is None:
            records.append(EpochMeta(
                filepath=str(fp), subject_id=subject_id,
                gender=gender, label=label, raw_label=raw_label,
                n_epochs=0, n_channels=0, sfreq=0.0,
                tmin=0.0, tmax=0.0, epoch_len_sec=0.0,
                ch_names=[], error=err,
            ))
            continue

        info      = epochs.info
        sfreq     = float(info["sfreq"])
        tmin      = float(epochs.tmin)
        tmax      = float(epochs.tmax)

        records.append(EpochMeta(
            filepath=str(fp),
            subject_id=subject_id,
            gender=gender,
            label=label,
            raw_label=raw_label,
            n_epochs=len(epochs),
            n_channels=len(info["ch_names"]),
            sfreq=sfreq,
            tmin=tmin,
            tmax=tmax,
            epoch_len_sec=round(tmax - tmin, 6),
            ch_names=list(info["ch_names"]),
        ))

    return records


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — Report
# ─────────────────────────────────────────────────────────────────────────────

def _line(char: str = "═", w: int = 68) -> None:
    print(char * w)


def print_report(records: list[EpochMeta]) -> None:
    """Print all six diagnostic answers."""

    _line()
    print("  MAAED (MOUNT ADORA EPILEPSY EEG) — FULL DIAGNOSTIC REPORT v2")
    _line()

    if not records:
        print("  No records. Check DATASET_DIR.")
        return

    ok     = [r for r in records if r.error is None]
    failed = [r for r in records if r.error is not None]

    print(f"\n  Files scanned      : {len(records)}")
    print(f"  Successfully loaded: {len(ok)}")
    print(f"  Failed to load     : {len(failed)}")

    if not ok:
        print("\n  ALL files failed. Check file integrity.")
        return

    # ── Q1: Epoch length ─────────────────────────────────────────────────
    _line("─")
    print("  Q1 ▶  EPOCH LENGTH (seconds)")
    _line("─")
    lengths = sorted(set(r.epoch_len_sec for r in ok))
    for L in lengths:
        n = sum(1 for r in ok if r.epoch_len_sec == L)
        print(f"  {L:.6f} s  ←  {n} file(s)   "
              f"[= {L:.4f} s = {int(round(L * 128))} samples @ 128 Hz]")
    if len(lengths) == 1:
        print(f"\n  ✔  Uniform epoch length: {lengths[0]:.4f} s")
    else:
        print("\n  ✘  Inconsistent epoch lengths — must standardize before training.")

    # ── Q2: Channel count breakdown ──────────────────────────────────────
    _line("─")
    print("  Q2 ▶  CHANNEL COUNT — WHICH SUBJECTS HAVE 20 CHANNELS?")
    _line("─")
    ch_counts = sorted(set(r.n_channels for r in ok))
    for nc in ch_counts:
        subset = [r for r in ok if r.n_channels == nc]
        print(f"\n  {nc} channels  ←  {len(subset)} file(s)")
        for r in subset:
            print(f"    {Path(r.filepath).name}")

    # ── Q3: Channel names — find the extra channel ────────────────────────
    _line("─")
    print("  Q3 ▶  CHANNEL NAMES — IDENTIFYING THE EXTRA CHANNEL")
    _line("─")

    # Collect channel sets per count
    ch_sets: dict[int, list[set[str]]] = {}
    for r in ok:
        nc = r.n_channels
        ch_sets.setdefault(nc, []).append(set(r.ch_names))

    # Find the common set for 19-channel files
    if 19 in ch_sets:
        base_set = ch_sets[19][0]
        for s in ch_sets[19][1:]:
            base_set = base_set & s
        print(f"\n  Channels present in ALL 19-channel files ({len(base_set)}):")
        print(f"  {sorted(base_set)}")
    else:
        base_set = set()

    # Find extra channels in 20-channel files
    if 20 in ch_sets:
        print(f"\n  Extra channel(s) in 20-channel files:")
        for s in ch_sets[20]:
            extra = s - base_set
            print(f"    Extra: {sorted(extra)}")
            print(f"    Full : {sorted(s)}")

    # Full channel list from first valid 19-channel file
    ref_19 = next((r for r in ok if r.n_channels == 19), None)
    if ref_19:
        print(f"\n  Reference 19-channel list (from {Path(ref_19.filepath).name}):")
        print(f"  {ref_19.ch_names}")

    # Full channel list from first valid 20-channel file
    ref_20 = next((r for r in ok if r.n_channels == 20), None)
    if ref_20:
        print(f"\n  Reference 20-channel list (from {Path(ref_20.filepath).name}):")
        print(f"  {ref_20.ch_names}")

    # ── Q4: Epileptic : Normal ratio ──────────────────────────────────────
    _line("─")
    print("  Q4 ▶  EPILEPTIC vs NORMAL EPOCH COUNTS (label assumption:")
    print("        positive=epileptic, negative=normal — verify below)")
    _line("─")

    df = pd.DataFrame([{
        "subject_id": r.subject_id,
        "gender":     r.gender,
        "label":      r.label,
        "raw_label":  r.raw_label,
        "n_epochs":   r.n_epochs,
        "n_channels": r.n_channels,
    } for r in ok])

    pivot = df.pivot_table(
        index="subject_id",
        columns="label",
        values="n_epochs",
        aggfunc="sum",
        fill_value=0,
    )

    # Add ratio column
    if "epileptic" in pivot.columns and "normal" in pivot.columns:
        pivot["ratio_epi_norm"] = (
            pivot["epileptic"] / pivot["normal"].replace(0, float("nan"))
        ).round(3)

    print()
    print(pivot.to_string())

    # Global stats
    print()
    _line("─")
    print("  GLOBAL EPOCH TOTALS")
    _line("─")
    for lbl in ["epileptic", "normal"]:
        if lbl in pivot.columns:
            total = pivot[lbl].sum()
            mn    = pivot[lbl].min()
            mx    = pivot[lbl].max()
            med   = pivot[lbl].median()
            print(f"\n  [{lbl}]")
            print(f"    Total  : {total:6d}")
            print(f"    Min    : {mn:6d}")
            print(f"    Max    : {mx:6d}")
            print(f"    Median : {med:.0f}")

    if "epileptic" in pivot.columns and "normal" in pivot.columns:
        n_epi  = pivot["epileptic"].sum()
        n_norm = pivot["normal"].sum()
        ratio  = n_epi / max(n_norm, 1)
        print(f"\n  Global ratio (epileptic:normal) = {ratio:.3f}")
        print(f"  Dominant class                 = "
              f"{'epileptic' if ratio > 1 else 'normal'}")

    # ── Q5: Zero-epoch subjects ───────────────────────────────────────────
    _line("─")
    print("  Q5 ▶  SUBJECTS WITH ZERO EPOCHS IN ONE OR BOTH CLASSES")
    _line("─")

    zero_issues: list[dict] = []
    for sid in pivot.index:
        row = pivot.loc[sid]
        for col in pivot.columns:
            if col == "ratio_epi_norm":
                continue
            if row[col] == 0:
                zero_issues.append({
                    "subject_id": sid,
                    "missing_class": col,
                    "other_class_epochs": int(
                        row[[c for c in pivot.columns
                             if c != col and c != "ratio_epi_norm"][0]]
                    ) if len([c for c in pivot.columns
                               if c != col and c != "ratio_epi_norm"]) > 0
                    else 0,
                })

    if zero_issues:
        print(f"\n  {len(zero_issues)} subject-class pair(s) with ZERO epochs:\n")
        print(f"  {'Subject':<20} {'Missing Class':<18} {'Other Class Epochs':>20}")
        print(f"  {'─'*20} {'─'*18} {'─'*20}")
        for z in zero_issues:
            print(f"  {z['subject_id']:<20} {z['missing_class']:<18} "
                  f"{z['other_class_epochs']:>20}")
        print()
        print(f"  ⚠  These subjects are single-class only.")
        print(f"     For LOSO: they can only train one side of the VAE.")
        print(f"     Decision required: include or exclude from evaluation?")
    else:
        print("\n  ✔  No subjects with zero epochs in any class.")

    # ── Q6: Per-subject channel count ────────────────────────────────────
    _line("─")
    print("  Q6 ▶  PER-SUBJECT CHANNEL COUNT (all files)")
    _line("─")

    ch_df = pd.DataFrame([{
        "subject_id": r.subject_id,
        "label":      r.label,
        "filename":   Path(r.filepath).name,
        "n_channels": r.n_channels,
        "ch_names":   "|".join(r.ch_names),
    } for r in ok])

    # Pivot: subject × label → channel count
    ch_pivot = ch_df.pivot_table(
        index="subject_id",
        columns="label",
        values="n_channels",
        aggfunc="max",
        fill_value=0,
    )
    print()
    print(ch_pivot.to_string())

    # Flag subjects where channel count differs between label files
    print()
    _line("─")
    print("  SUBJECTS WITH INCONSISTENT CHANNEL COUNT ACROSS LABEL FILES")
    _line("─")
    inconsistent = []
    for sid in ch_df["subject_id"].unique():
        sub = ch_df[ch_df["subject_id"] == sid]
        counts = sub["n_channels"].unique()
        if len(counts) > 1:
            inconsistent.append((sid, sorted(counts)))
    if inconsistent:
        for sid, counts in inconsistent:
            print(f"  {sid}: channel counts = {counts}")
    else:
        print("\n  ✔  All subjects have consistent channel count across files.")

    # ── Failed files ──────────────────────────────────────────────────────
    if failed:
        _line("─")
        print(f"  FAILED FILES ({len(failed)})")
        _line("─")
        for r in failed:
            print(f"  {Path(r.filepath).name}")
            print(f"    → {r.error}")

    _line()


# ─────────────────────────────────────���───────────────────────────────────────
# SECTION 5 — CSV Export
# ─────────────────────────────────────────────────────────────────────────────

def export_csv(records: list[EpochMeta]) -> None:
    """Export full per-file metadata to CSV."""
    rows = [{
        "filepath":      r.filepath,
        "subject_id":    r.subject_id,
        "gender":        r.gender,
        "label":         r.label,
        "raw_label":     r.raw_label,
        "n_epochs":      r.n_epochs,
        "n_channels":    r.n_channels,
        "sfreq":         r.sfreq,
        "tmin":          r.tmin,
        "tmax":          r.tmax,
        "epoch_len_sec": r.epoch_len_sec,
        "ch_names":      "|".join(r.ch_names),
        "error":         r.error or "",
    } for r in records]

    out = Path("maaed_audit_v2.csv")
    pd.DataFrame(rows).to_csv(out, index=False)
    print(f"\n  ✔  Full audit saved to: {out.resolve()}\n")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — Entrypoint
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    _line()
    print("  MAAED DATASET AUDIT v2 — EPILEPSY DIAGNOSTIC")
    _line()

    if not DATASET_DIR.exists():
        print(f"\n  ✘  DATASET_DIR not found:\n    {DATASET_DIR}")
        print("  Fix the path at the top of this script.")
        return

    records = scan_dataset(DATASET_DIR)

    if records:
        print_report(records)
        export_csv(records)


if __name__ == "__main__":
    main()

════════════════════════════════════════════════════════════════════
  MAAED DATASET AUDIT v2 — EPILEPSY DIAGNOSTIC
════════════════════════════════════════════════════════════════════

  Files found : 212
  Scanning...  (this may take 1-2 minutes)

════════════════════════════════════════════════════════════════════
  MAAED (MOUNT ADORA EPILEPSY EEG) — FULL DIAGNOSTIC REPORT v2
════════════════════════════════════════════════════════════════════

  Files scanned      : 212
  Successfully loaded: 212
  Failed to load     : 0
────────────────────────────────────────────────────────────────────
  Q1 ▶  EPOCH LENGTH (seconds)
────────────────────────────────────────────────────────────────────
  3.992188 s  ←  212 file(s)   [= 3.9922 s = 511 samples @ 128 Hz]

  ✔  Uniform epoch length: 3.9922 s
────────────────────────────────────────────────────────────────────
  Q2 ▶  CHANNEL COUNT — WHICH SUBJECTS HAVE 20 CHANNELS?
────────────────────────────────────────────────────────────────────



In [4]:
"""
β-VAE Epilepsy Anomaly Detection Pipeline
==========================================
Dataset   : MAAED (Mount Adora Epilepsy EEG Dataset)
Task      : Unsupervised seizure detection via reconstruction error
Method    : β-VAE trained on NORMAL epochs only
            Anomaly score = per-epoch MSE(input, reconstruction)
Validation: Leave-One-Subject-Out (LOSO) — strict, zero leakage

FIX v2    : N_SAMPLES corrected 511 → 512 (files contain 512 samples)
            Decoder crop layer removed (512 is power-of-2, no crop needed)

Input shape : (batch, 19, 512)
Latent dim  : 64
Beta        : 4

Dependencies:
    pip install mne numpy pandas scikit-learn tensorflow

Seed : 42
Date : 2026-03-12
"""

from __future__ import annotations

import os
import re
import time
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import mne
import tensorflow as tf
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    f1_score,
    confusion_matrix,
)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
#  ▶  CONFIG
# ─────────────────────────────────────────────────────────────────────────────
DATASET_DIR  = Path(r"/kaggle/input/datasets/johanliebert28/maaed-cleaned-epochs-archive-new")
RESULTS_DIR  = Path("./vae_results")

# EEG constants — CONFIRMED from audit + shape mismatch fix
N_CHANNELS    = 19
N_SAMPLES     = 512          # ← FIXED: was 511, actual data is 512 samples
SFREQ         = 128.0        # Hz
KEEP_CHANNELS = [            # canonical 19-channel order
    "Fp1","Fp2","F7","F3","Fz","F4","F8",
    "T3","C3","Cz","C4","T4",
    "T5","P3","Pz","P4","T6","O1","O2",
]

# Model hyperparameters
LATENT_DIM    = 64
BETA          = 4.0
EPOCHS_TRAIN  = 50
BATCH_SIZE    = 32
LEARNING_RATE = 1e-3

# Single-class subjects — excluded from LOSO eval, used for training only
TRAIN_ONLY_SUBJECTS = {
    "female_51",                          # normal only
    "male_52","male_53","male_54","male_55","male_56",
    "male_57","male_58","male_59","male_60",
}
# ─────────────────────────────────────────────────────────────────────────────

RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — Data Loading
# ─────────────────────────────────────────────────────────────────────────────

def _parse_filename(filename: str) -> tuple[str, str, str]:
    """
    Parse: {gender}_{valence}_raw_{id}_clean-epo.fif

    CORRECTED LABEL MAPPING (confirmed from AUROC=0.0 inversion):
      positive → 'normal'     (healthy subjects)
      negative → 'epileptic'  (seizure subjects)

    Returns (subject_id, label, gender)
    """
    name  = filename.lower()
    match = re.match(
        r"^(male|female)_(positive|negative)_raw_(\d+)_",
        name,
    )
    if not match:
        return "unknown", "unknown", "unknown"

    gender     = match.group(1)
    raw_label  = match.group(2)
    subject_id = f"{gender}_{match.group(3)}"

    # CORRECTED: positive=normal, negative=epileptic
    label = "normal" if raw_label == "positive" else "epileptic"
    return subject_id, label, gender

def load_subject_epochs(
    fif_path:      Path,
    keep_channels: list[str] = KEEP_CHANNELS,
    n_channels:    int       = N_CHANNELS,
    n_samples:     int       = N_SAMPLES,
) -> Optional[np.ndarray]:
    """
    Load one .fif file → numpy array (N, 19, 512).

    Steps
    -----
    1. Read epochs with MNE
    2. Drop 'Oz' if present (4 affected subjects)
    3. Pick canonical 19 channels in fixed order
    4. Verify shape is (N, 19, 512) — skip file if not
    5. Per-epoch per-channel z-score normalisation
    6. Clip to [-6, 6] as artefact safety net

    Parameters
    ----------
    fif_path      : path to .fif file
    keep_channels : canonical channel list (19 channels)
    n_channels    : expected channel count (19)
    n_samples     : expected time samples  (512) ← FIXED

    Returns
    -------
    data : ndarray (N, 19, 512) float32, or None on failure
    """
    try:
        epochs = mne.read_epochs(str(fif_path), preload=True, verbose=False)
    except Exception as exc:
        print(f"  [LOAD ERROR] {fif_path.name}: {exc}")
        return None

    # Drop Oz if present
    if "Oz" in epochs.ch_names:
        epochs = epochs.drop_channels(["Oz"])

    # Verify all canonical channels are present
    missing = [ch for ch in keep_channels if ch not in epochs.ch_names]
    if missing:
        print(f"  [SKIP] {fif_path.name}: missing channels {missing}")
        return None

    # Pick in canonical order
    epochs = epochs.pick_channels(keep_channels, ordered=True)

    # Get data: (N, 19, T)
    data = epochs.get_data()

    # Shape guard — now expects 512
    if data.shape[1] != n_channels or data.shape[2] != n_samples:
        print(f"  [SHAPE MISMATCH] {fif_path.name}: {data.shape} "
              f"expected ({data.shape[0]}, {n_channels}, {n_samples})")
        return None

    # Per-epoch per-channel z-score: axis=2 is time
    mu  = data.mean(axis=2, keepdims=True)           # (N, 19, 1)
    std = data.std(axis=2,  keepdims=True) + 1e-8    # (N, 19, 1)
    data = (data - mu) / std

    # Clip artefact outliers
    data = np.clip(data, -6.0, 6.0).astype(np.float32)

    return data


def build_subject_registry(directory: Path) -> pd.DataFrame:
    """
    Scan directory → DataFrame with one row per .fif file.

    Columns: filepath, subject_id, label, gender, in_loso_pool
    """
    fif_files = sorted(set(
        list(directory.glob("**/*-epo.fif")) +
        list(directory.glob("**/*_epo.fif")) +
        list(directory.glob("**/*.fif"))
    ))

    rows = []
    for fp in fif_files:
        subject_id, label, gender = _parse_filename(fp.name)
        if subject_id == "unknown":
            continue
        rows.append({
            "filepath":     str(fp),
            "subject_id":   subject_id,
            "label":        label,
            "gender":       gender,
            "in_loso_pool": subject_id not in TRAIN_ONLY_SUBJECTS,
        })

    return pd.DataFrame(rows)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — β-VAE Model
# ─────────────────────────────────────────────────────────────────────────────

class Sampling(tf.keras.layers.Layer):
    """
    Reparameterisation trick.

    z = μ + σ·ε,  ε ~ N(0, I)

    Input  : [μ, log_σ]  each (batch, latent_dim)
    Output : z           (batch, latent_dim)
    """
    def call(self, inputs: list[tf.Tensor]) -> tf.Tensor:
        mu, log_sigma = inputs
        eps = tf.random.normal(shape=tf.shape(mu), seed=SEED)
        return mu + tf.exp(0.5 * log_sigma) * eps


def build_encoder(
    n_channels: int = N_CHANNELS,
    n_samples:  int = N_SAMPLES,
    latent_dim: int = LATENT_DIM,
) -> tf.keras.Model:
    """
    Encoder: (batch, 19, 512) → μ, log_σ ∈ R^64

    Architecture
    ────────────
    DepthwiseConv1D  kernel=25           spatial filter per channel
    PointwiseConv1D  filters=64          cross-channel mixing
    Permute          (batch,19,512)
                   → (batch,512,64)      time-first for GRU
    StrideConv1D ×3  stride=2            512→256→128→64
    BiGRU            units=128           (batch, 256)
    Dense            128, elu
    Dense → μ        latent_dim
    Dense → log_σ    latent_dim
    Sampling         z = μ + σ·ε

    I/O shapes
    ──────────
    Input  : (batch, 19, 512)
    Output : [μ (batch,64), log_σ (batch,64), z (batch,64)]
    """
    inp = tf.keras.Input(shape=(n_channels, n_samples), name="encoder_input")

    # Depthwise: (batch, 19, 512) → (batch, 19, 512)
    x = tf.keras.layers.DepthwiseConv1D(
        kernel_size=25, padding="same", activation="elu",
        name="depthwise_conv",
    )(inp)

    # Pointwise: (batch, 19, 512) → (batch, 64, 512)
    x = tf.keras.layers.Conv1D(
        filters=64, kernel_size=1, activation="elu",
        name="pointwise_conv",
    )(x)

    # Permute to time-first: (batch, 64, 512) → (batch, 512, 64)
    x = tf.keras.layers.Permute((2, 1), name="enc_permute")(x)

    # Strided downsampling: 512 → 256 → 128 → 64  (exact powers of 2)
    for i, filters in enumerate([64, 128, 128]):
        x = tf.keras.layers.Conv1D(
            filters=filters, kernel_size=5,
            strides=2, padding="same", activation="elu",
            name=f"stride_conv_{i}",
        )(x)
    # (batch, 64, 128)

    # BiGRU: (batch, 64, 128) → (batch, 256)
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.GRU(units=128, return_sequences=False),
        name="bigru",
    )(x)

    x  = tf.keras.layers.Dense(128, activation="elu", name="enc_dense")(x)
    mu      = tf.keras.layers.Dense(latent_dim, name="mu")(x)
    log_sig = tf.keras.layers.Dense(latent_dim, name="log_sigma")(x)
    z       = Sampling(name="z")([mu, log_sig])

    return tf.keras.Model(inp, [mu, log_sig, z], name="encoder")


def build_decoder(
    n_channels: int = N_CHANNELS,
    n_samples:  int = N_SAMPLES,
    latent_dim: int = LATENT_DIM,
) -> tf.keras.Model:
    """
    Decoder: z ∈ R^64 → (batch, 19, 512)

    Architecture
    ────────────
    Dense            64*128 → Reshape (batch, 64, 128)
    GRU              units=128, return_sequences=True
    ConvTranspose ×3 stride=2  64→128→256→512  (exact, no crop needed)
    Conv1D           filters=19, kernel=1  → 19 channels
    Permute          (batch,512,19) → (batch,19,512)

    NOTE: 512 = 2^9. Three stride-2 upsamples from 64 = 2^6 give
          exactly 512. No crop layer required. This is why N_SAMPLES
          must be 512 and not 511.

    I/O shapes
    ──────────
    Input  : z (batch, 64)
    Output : (batch, 19, 512)
    """
    z_inp = tf.keras.Input(shape=(latent_dim,), name="decoder_input")

    # Dense + reshape: (batch,64) → (batch,64,128)
    x = tf.keras.layers.Dense(64 * 128, activation="elu", name="dec_dense")(z_inp)
    x = tf.keras.layers.Reshape((64, 128), name="reshape")(x)

    # GRU: (batch,64,128) → (batch,64,128)
    x = tf.keras.layers.GRU(
        units=128, return_sequences=True, name="dec_gru",
    )(x)

    # Upsample: 64→128→256→512  (NO CROP — 512 is exact)
    for i, filters in enumerate([128, 64, 64]):
        x = tf.keras.layers.Conv1DTranspose(
            filters=filters, kernel_size=5,
            strides=2, padding="same", activation="elu",
            name=f"deconv_{i}",
        )(x)
    # (batch, 512, 64)

    # Map to 19 channels: (batch,512,64) → (batch,512,19)
    x = tf.keras.layers.Conv1D(
        filters=n_channels, kernel_size=1,
        activation="linear", name="output_conv",
    )(x)

    # Permute to channel-first: (batch,512,19) → (batch,19,512)
    x = tf.keras.layers.Permute((2, 1), name="output_permute")(x)

    return tf.keras.Model(z_inp, x, name="decoder")


class BetaVAE(tf.keras.Model):
    """
    β-VAE for EEG anomaly detection.

    Loss = MSE(x, x̂) + β · KL(q(z|x) ‖ N(0,I))

    Anomaly score at test time = MSE(x, x̂) per epoch.
    Higher score = more anomalous = more likely epileptic.
    """

    def __init__(
        self,
        encoder: tf.keras.Model,
        decoder: tf.keras.Model,
        beta:    float = BETA,
    ) -> None:
        super().__init__()
        self.encoder      = encoder
        self.decoder      = decoder
        self.beta         = beta
        self._recon_loss  = tf.keras.metrics.Mean(name="recon_loss")
        self._kl_loss     = tf.keras.metrics.Mean(name="kl_loss")
        self._total_loss  = tf.keras.metrics.Mean(name="total_loss")

    @property
    def metrics(self) -> list:
        return [self._total_loss, self._recon_loss, self._kl_loss]

    def call(
        self, x: tf.Tensor, training: bool = False
    ) -> tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
        """
        Forward pass.

        Input  : x      (batch, 19, 512)
        Output : x_hat  (batch, 19, 512)
                 mu     (batch, 64)
                 log_sig(batch, 64)
        """
        mu, log_sig, z = self.encoder(x, training=training)
        x_hat          = self.decoder(z, training=training)
        return x_hat, mu, log_sig

    def _compute_loss(
        self, x: tf.Tensor
    ) -> tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
        """
        ELBO = reconstruction_loss + β · KL

        reconstruction_loss : mean over batch of sum over (channels × time)
        KL                  : mean over batch of sum over latent dims
        """
        x_hat, mu, log_sig = self(x, training=True)

        recon = tf.reduce_mean(
            tf.reduce_sum(tf.square(x - x_hat), axis=[1, 2])
        )
        kl = -0.5 * tf.reduce_mean(
            tf.reduce_sum(
                1.0 + log_sig - tf.square(mu) - tf.exp(log_sig),
                axis=1,
            )
        )
        total = recon + self.beta * kl
        return total, recon, kl

    def train_step(self, data: tf.Tensor) -> dict:
        with tf.GradientTape() as tape:
            total, recon, kl = self._compute_loss(data)
        grads = tape.gradient(total, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self._total_loss.update_state(total)
        self._recon_loss.update_state(recon)
        self._kl_loss.update_state(kl)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data: tf.Tensor) -> dict:
        total, recon, kl = self._compute_loss(data)
        self._total_loss.update_state(total)
        self._recon_loss.update_state(recon)
        self._kl_loss.update_state(kl)
        return {m.name: m.result() for m in self.metrics}

    def anomaly_score(self, x: np.ndarray) -> np.ndarray:
        """
        Per-epoch reconstruction MSE — the anomaly score.

        Parameters
        ----------
        x : (N, 19, 512) float32

        Returns
        -------
        scores : (N,) float32   higher = more anomalous = likely epileptic
        """
        x_t    = tf.constant(x, dtype=tf.float32)
        x_hat, _, _ = self(x_t, training=False)
        scores = tf.reduce_mean(
            tf.square(x_t - x_hat), axis=[1, 2]
        ).numpy()
        return scores.astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — Metrics
# ─────────────────────────────────────────────────────────────────────────────

def sensitivity_at_specificity(
    y_true:             np.ndarray,
    scores:             np.ndarray,
    target_specificity: float = 0.95,
) -> float:
    """
    Sensitivity at the ROC threshold that achieves target_specificity.

    Parameters
    ----------
    y_true             : (N,) int — 1=epileptic, 0=normal
    scores             : (N,) float — anomaly score
    target_specificity : desired specificity on normal class

    Returns
    -------
    sensitivity (float) or nan if target unreachable
    """
    fpr, tpr, _ = roc_curve(y_true, scores)
    specificities = 1.0 - fpr
    idx = np.where(specificities >= target_specificity)[0]
    if len(idx) == 0:
        return float("nan")
    best = idx[np.argmin(np.abs(specificities[idx] - target_specificity))]
    return float(tpr[best])


def compute_metrics(
    y_true:     np.ndarray,
    scores:     np.ndarray,
    latency_ms: float,
    n_params:   int,
) -> dict:
    """
    Compute all required metrics for one LOSO fold.

    Threshold for F1/confusion matrix: median of anomaly scores.
    This is the standard threshold-free approach for anomaly detectors.
    """
    auroc    = roc_auc_score(y_true, scores)
    thresh   = np.median(scores)
    y_pred   = (scores >= thresh).astype(int)
    f1       = f1_score(y_true, y_pred, zero_division=0)
    sens_95  = sensitivity_at_specificity(y_true, scores, 0.95)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity    = tp / max(tp + fn, 1)
    specificity    = tn / max(tn + fp, 1)

    return {
        "auroc":          round(float(auroc),       4),
        "f1":             round(float(f1),           4),
        "sensitivity":    round(float(sensitivity),  4),
        "specificity":    round(float(specificity),  4),
        "sens_at_95spec": round(float(sens_95),      4),
        "latency_ms":     round(float(latency_ms),   4),
        "n_params":       int(n_params),
    }


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — LOSO Training Loop
# ─────────────────────────────────────────────────────────────────────────────

def run_loso(registry: pd.DataFrame) -> pd.DataFrame:
    """
    Leave-One-Subject-Out cross-validation.

    Protocol (zero-leakage guarantee)
    ──────────────────────────────────
    For each test subject S in loso_pool (101 subjects):
      Train  → ALL normal epochs from every subject EXCEPT S
      Test   → ALL epochs (normal + epileptic) from S only
      Never  → epileptic epochs enter training at any point

    Parameters
    ----------
    registry : output of build_subject_registry()

    Returns
    -------
    results_df : one row per completed fold
    """
    loso_subjects = sorted(
        registry[registry["in_loso_pool"]]["subject_id"].unique()
    )
    print(f"\n  LOSO pool : {len(loso_subjects)} subjects")
    print(f"  Pre-loading all epochs into memory...")

    # Cache: (subject_id, label) → ndarray (N, 19, 512)
    epoch_cache: dict[tuple[str, str], np.ndarray] = {}
    skipped = 0

    for _, row in registry.iterrows():
        key  = (row["subject_id"], row["label"])
        data = load_subject_epochs(Path(row["filepath"]))
        if data is not None:
            epoch_cache[key] = data
        else:
            skipped += 1

    print(f"  Loaded {len(epoch_cache)} arrays  |  Skipped {skipped} files\n")

    if len(epoch_cache) == 0:
        print("  [FATAL] No data loaded. Pipeline cannot proceed.")
        return pd.DataFrame()

    all_results = []

    for fold_idx, test_subject in enumerate(loso_subjects):
        print(f"  Fold {fold_idx+1:03d}/{len(loso_subjects)} "
              f"| Test: {test_subject}")

        # ── Build training set ───────────────────────────────────────────────
        train_arrays = [
            data
            for (sid, label), data in epoch_cache.items()
            if sid != test_subject and label == "normal"
        ]

        if not train_arrays:
            print(f"    [SKIP] No training data found.")
            continue

        X_train = np.concatenate(train_arrays, axis=0)   # (N_train, 19, 512)
        np.random.shuffle(X_train)
        print(f"    Train normal epochs : {len(X_train)}")

        # ── Build test set ──────────────────────────────────────────��────────
        X_parts, y_parts = [], []

        normal_data    = epoch_cache.get((test_subject, "normal"))
        epileptic_data = epoch_cache.get((test_subject, "epileptic"))

        if normal_data is not None and len(normal_data) > 0:
            X_parts.append(normal_data)
            y_parts.append(np.zeros(len(normal_data), dtype=np.int32))

        if epileptic_data is not None and len(epileptic_data) > 0:
            X_parts.append(epileptic_data)
            y_parts.append(np.ones(len(epileptic_data), dtype=np.int32))

        if len(X_parts) < 2:
            print(f"    [SKIP] Test subject missing one class.")
            continue

        X_test = np.concatenate(X_parts, axis=0)    # (N_test, 19, 512)
        y_test = np.concatenate(y_parts, axis=0)    # (N_test,)
        print(f"    Test  normal={int((y_test==0).sum())}  "
              f"epileptic={int((y_test==1).sum())}")

        # ── Build + train VAE ────────────────────────────────────────────────
        tf.keras.backend.clear_session()
        tf.random.set_seed(SEED)

        vae = BetaVAE(build_encoder(), build_decoder(), beta=BETA)
        vae.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE))

        train_ds = (
            tf.data.Dataset.from_tensor_slices(X_train)
            .shuffle(buffer_size=min(len(X_train), 5000), seed=SEED)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE)
        )

        vae.fit(
            train_ds,
            epochs=EPOCHS_TRAIN,
            callbacks=[
                tf.keras.callbacks.EarlyStopping(
                    monitor="total_loss",
                    patience=5,
                    mode="min",
                    restore_best_weights=True,
                    verbose=0,
                )
            ],
            verbose=0,
        )

        # ── Anomaly scoring + latency ────────────────────────────────────────
        n_params = int(sum(np.prod(v.shape) for v in vae.trainable_variables))

        _ = vae.anomaly_score(X_test[:1])           # warm-up

        t0     = time.perf_counter()
        scores = vae.anomaly_score(X_test)           # (N_test,)
        t1     = time.perf_counter()
        latency_ms = ((t1 - t0) / len(X_test)) * 1000

        # ── Metrics ──────────────────────────────────────────────────────────
        if len(np.unique(y_test)) < 2:
            print(f"    [SKIP] Only one class in test set.")
            continue

        metrics = compute_metrics(y_test, scores, latency_ms, n_params)
        metrics.update({
            "test_subject":     test_subject,
            "n_train_normal":   len(X_train),
            "n_test_normal":    int((y_test == 0).sum()),
            "n_test_epileptic": int((y_test == 1).sum()),
            "fold":             fold_idx + 1,
        })
        all_results.append(metrics)

        print(f"    AUROC={metrics['auroc']:.4f}  "
              f"F1={metrics['f1']:.4f}  "
              f"Sens@95Spec={metrics['sens_at_95spec']:.4f}  "
              f"Latency={latency_ms:.2f}ms")

    return pd.DataFrame(all_results)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — Final Report
# ─────────────────────────────────────────────────────────────────────────────

def print_final_report(results: pd.DataFrame) -> None:
    """Mean ± std across all folds. Explicit hypothesis pass/fail."""
    print("\n" + "═" * 68)
    print("  LOSO FINAL RESULTS — β-VAE EPILEPSY ANOMALY DETECTION")
    print("═" * 68)

    for m in ["auroc","f1","sensitivity","specificity",
              "sens_at_95spec","latency_ms"]:
        if m not in results.columns:
            continue
        col  = results[m].dropna()
        print(f"  {m:<22}: {col.mean():.4f} ± {col.std():.4f}")

    auroc_mean = results["auroc"].mean()
    print("\n" + "─" * 68)
    if auroc_mean >= 0.85:
        print(f"  ✔  HYPOTHESIS MET   : mean AUROC = {auroc_mean:.4f} ≥ 0.85")
    else:
        print(f"  ✘  HYPOTHESIS FAILED: mean AUROC = {auroc_mean:.4f} < 0.85")
        print("     Next steps: increase β, latent dim, or training epochs.")

    print(f"\n  n_params       : {int(results['n_params'].iloc[0]):,}")
    print(f"  Folds completed: {len(results)}")
    print("═" * 68)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — Entrypoint
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    print("═" * 68)
    print("  β-VAE EPILEPSY DETECTION PIPELINE  v2")
    print(f"  Seed={SEED}  |  LOSO  |  N_SAMPLES={N_SAMPLES}  "
          f"|  N_CHANNELS={N_CHANNELS}")
    print("═" * 68)

    if not DATASET_DIR.exists():
        print(f"\n  ✘ DATASET_DIR not found: {DATASET_DIR}")
        return

    registry = build_subject_registry(DATASET_DIR)
    loso_n   = registry[registry["in_loso_pool"]]["subject_id"].nunique()
    total_n  = registry["subject_id"].nunique()

    print(f"\n  Total subjects      : {total_n}")
    print(f"  LOSO eval subjects  : {loso_n}")
    print(f"  Train-only subjects : {total_n - loso_n}")

    results = run_loso(registry)

    if results.empty:
        print("\n  [ERROR] No folds completed.")
        return

    out_csv = RESULTS_DIR / "loso_results.csv"
    results.to_csv(out_csv, index=False)
    print(f"\n  ✔  Results saved: {out_csv.resolve()}")

    print_final_report(results)


if __name__ == "__main__":
    main()

════════════════════════════════════════════════════════════════════
  β-VAE EPILEPSY DETECTION PIPELINE  v2
  Seed=42  |  LOSO  |  N_SAMPLES=512  |  N_CHANNELS=19
════════════════════════════════════════════════════════════════════

  Total subjects      : 111
  LOSO eval subjects  : 101
  Train-only subjects : 10

  LOSO pool : 101 subjects
  Pre-loading all epochs into memory...
  Loaded 212 arrays  |  Skipped 0 files

  Fold 001/101 | Test: female_1
    Train normal epochs : 24071
    Test  normal=369  epileptic=142
    AUROC=1.0000  F1=0.7136  Sens@95Spec=0.0000  Latency=0.13ms
  Fold 002/101 | Test: female_10
    Train normal epochs : 24221
    Test  normal=219  epileptic=183
    AUROC=1.0000  F1=0.9531  Sens@95Spec=0.0000  Latency=0.16ms
  Fold 003/101 | Test: female_11
    Train normal epochs : 24345
    Test  normal=95  epileptic=112


KeyboardInterrupt: 

In [5]:
"""
Raw score distribution check — run BEFORE resuming LOSO.
Checks whether amplitude confound or normalisation failure
is causing trivial AUROC=1.0.

Run after fixing label mapping.
Date: 2026-03-12
"""

from __future__ import annotations
import warnings
from pathlib import Path
import numpy as np
import mne

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore")

DATASET_DIR = Path(
    r"/kaggle/input/datasets/johanliebert28/maaed-cleaned-epochs-archive-new"
)
KEEP_CHANNELS = [
    "Fp1","Fp2","F7","F3","Fz","F4","F8",
    "T3","C3","Cz","C4","T4",
    "T5","P3","Pz","P4","T6","O1","O2",
]

def load_raw(fif_path: Path) -> np.ndarray | None:
    """Load epochs WITHOUT normalisation — raw microvolts."""
    try:
        epochs = mne.read_epochs(str(fif_path), preload=True, verbose=False)
        if "Oz" in epochs.ch_names:
            epochs = epochs.drop_channels(["Oz"])
        missing = [c for c in KEEP_CHANNELS if c not in epochs.ch_names]
        if missing:
            return None
        epochs = epochs.pick_channels(KEEP_CHANNELS, ordered=True)
        return epochs.get_data().astype(np.float32)   # (N, 19, 512) RAW
    except Exception:
        return None


def check_amplitude_confound(directory: Path, n_subjects: int = 5) -> None:
    """
    Load first n_subjects of each class.
    Compare raw amplitude (RMS) between normal and epileptic.
    If RMS differs by >2x → amplitude confound confirmed.
    """
    print("\n" + "═"*60)
    print("  RAW AMPLITUDE CONFOUND CHECK (no normalisation)")
    print("═"*60)

    normal_rms_all    : list[float] = []
    epileptic_rms_all : list[float] = []

    normal_files    = sorted(directory.glob("**/female_positive_raw_*_clean-epo.fif"))
    epileptic_files = sorted(directory.glob("**/female_negative_raw_*_clean-epo.fif"))

    print(f"\n  Checking first {n_subjects} subjects per class...\n")

    for fp in normal_files[:n_subjects]:
        data = load_raw(fp)
        if data is not None:
            rms = float(np.sqrt(np.mean(data ** 2)))
            normal_rms_all.append(rms)
            print(f"  NORMAL    {fp.name:<45} RMS={rms:.6f} V")

    for fp in epileptic_files[:n_subjects]:
        data = load_raw(fp)
        if data is not None:
            rms = float(np.sqrt(np.mean(data ** 2)))
            epileptic_rms_all.append(rms)
            print(f"  EPILEPTIC {fp.name:<45} RMS={rms:.6f} V")

    print()
    print("─"*60)
    mean_n = np.mean(normal_rms_all)
    mean_e = np.mean(epileptic_rms_all)
    ratio  = mean_n / (mean_e + 1e-12)

    print(f"  Mean RMS NORMAL    : {mean_n:.8f} V")
    print(f"  Mean RMS EPILEPTIC : {mean_e:.8f} V")
    print(f"  Ratio (N/E)        : {ratio:.4f}")
    print()

    if abs(ratio - 1.0) > 0.5:
        print("  ✘  AMPLITUDE CONFOUND CONFIRMED")
        print("     The two classes have different raw signal amplitudes.")
        print("     Z-score normalisation is masking this but VAE still")
        print("     exploits residual amplitude differences.")
        print("     Fix: global-subject normalisation before training.")
    else:
        print("  ✔  No amplitude confound. Look elsewhere for AUROC=1.0 cause.")

    # ── Check score variance (zero-variance = all scores identical) ───────
    print()
    print("─"*60)
    print("  NORMALISATION VARIANCE CHECK")
    print("─"*60)
    print("  Loading 10 epochs each class WITH z-score normalisation...")

    def load_normalised(fp: Path) -> np.ndarray | None:
        data = load_raw(fp)
        if data is None:
            return None
        mu  = data.mean(axis=2, keepdims=True)
        std = data.std(axis=2,  keepdims=True) + 1e-8
        return np.clip((data - mu) / std, -6.0, 6.0)

    n_sample  = load_normalised(normal_files[0])
    e_sample  = load_normalised(epileptic_files[0])

    if n_sample is not None and e_sample is not None:
        print(f"\n  NORMAL    epoch variance : {n_sample.var():.8f}")
        print(f"  EPILEPTIC epoch variance : {e_sample.var():.8f}")

        n_mean_power = float(np.mean(n_sample ** 2))
        e_mean_power = float(np.mean(e_sample ** 2))
        print(f"\n  NORMAL    mean power after norm: {n_mean_power:.6f}")
        print(f"  EPILEPTIC mean power after norm: {e_mean_power:.6f}")

        if abs(n_mean_power - e_mean_power) < 0.01:
            print("\n  ✔  Power equalised after normalisation.")
            print("     AUROC=1.0 is NOT from amplitude. Check spectral content.")
        else:
            print("\n  ✘  Power NOT equalised after normalisation.")
            print("     VAE is exploiting power difference. Fix normalisation.")

    print("═"*60)


if __name__ == "__main__":
    check_amplitude_confound(DATASET_DIR, n_subjects=5)


════════════════════════════════════════════════════════════
  RAW AMPLITUDE CONFOUND CHECK (no normalisation)
════════════════════════════════════════════════════════════

  Checking first 5 subjects per class...

  NORMAL    female_positive_raw_10_clean-epo.fif          RMS=0.000052 V
  NORMAL    female_positive_raw_11_clean-epo.fif          RMS=0.000017 V
  NORMAL    female_positive_raw_12_clean-epo.fif          RMS=0.000016 V
  NORMAL    female_positive_raw_13_clean-epo.fif          RMS=0.000026 V
  NORMAL    female_positive_raw_14_clean-epo.fif          RMS=0.000066 V
  EPILEPTIC female_negative_raw_10_clean-epo.fif          RMS=0.000005 V
  EPILEPTIC female_negative_raw_11_clean-epo.fif          RMS=0.000007 V
  EPILEPTIC female_negative_raw_12_clean-epo.fif          RMS=0.000006 V
  EPILEPTIC female_negative_raw_13_clean-epo.fif          RMS=0.000009 V
  EPILEPTIC female_negative_raw_14_clean-epo.fif          RMS=0.000006 V

─────────────────────────────────────────────────────

In [6]:
"""
Spectral Confound & Leakage Diagnostic
=======================================
Checks whether AUROC=1.0 comes from:
  1. Spectral power differences (delta/theta/alpha/beta bands)
  2. Recording-level confounds (line noise, DC offset)
  3. Subject identity leakage via metadata

Run this BEFORE resuming LOSO.
Date: 2026-03-12
"""

from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import mne
from scipy import signal as sp_signal
from scipy.stats import mannwhitneyu

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore")

DATASET_DIR = Path(
    r"/kaggle/input/datasets/johanliebert28/maaed-cleaned-epochs-archive-new"
)
KEEP_CHANNELS = [
    "Fp1","Fp2","F7","F3","Fz","F4","F8",
    "T3","C3","Cz","C4","T4",
    "T5","P3","Pz","P4","T6","O1","O2",
]
SFREQ      = 128.0
N_SAMPLES  = 512
FREQ_BANDS = {
    "delta": (0.5,  4.0),
    "theta": (4.0,  8.0),
    "alpha": (8.0, 13.0),
    "beta":  (13.0, 30.0),
    "gamma": (30.0, 45.0),
}
N_SUBJECTS_PER_CLASS = 10   # subjects to sample per class


# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def load_normalised(fif_path: Path) -> np.ndarray | None:
    """
    Load and z-score normalise epochs.
    Returns (N, 19, 512) float32 or None.
    """
    try:
        epochs = mne.read_epochs(str(fif_path), preload=True, verbose=False)
    except Exception:
        return None

    if "Oz" in epochs.ch_names:
        epochs = epochs.drop_channels(["Oz"])

    missing = [c for c in KEEP_CHANNELS if c not in epochs.ch_names]
    if missing:
        return None

    epochs = epochs.pick_channels(KEEP_CHANNELS, ordered=True)
    data   = epochs.get_data().astype(np.float32)   # (N, 19, 512)

    if data.shape[1] != 19 or data.shape[2] != 512:
        return None

    # Per-epoch per-channel z-score (same as pipeline)
    mu  = data.mean(axis=2, keepdims=True)
    std = data.std(axis=2,  keepdims=True) + 1e-8
    data = np.clip((data - mu) / std, -6.0, 6.0)

    return data


def bandpower(
    data:   np.ndarray,    # (N, 19, 512)
    sfreq:  float,
    fmin:   float,
    fmax:   float,
) -> np.ndarray:
    """
    Mean band power across all epochs and channels.

    Uses Welch PSD. Returns (N,) array — one value per epoch.

    Parameters
    ----------
    data  : (N, 19, 512) normalised EEG
    sfreq : sampling frequency
    fmin  : band low edge Hz
    fmax  : band high edge Hz

    Returns
    -------
    power : (N,) float32 — mean band power per epoch
    """
    freqs, psd = sp_signal.welch(
        data,
        fs=sfreq,
        nperseg=min(256, N_SAMPLES),
        axis=2,                         # compute over time axis
    )
    # psd shape: (N, 19, n_freqs)
    freq_mask = (freqs >= fmin) & (freqs <= fmax)
    band_psd  = psd[:, :, freq_mask].mean(axis=(1, 2))  # (N,)
    return band_psd.astype(np.float32)


def load_class_data(
    directory:  Path,
    class_glob: str,
    n_subjects: int,
) -> np.ndarray | None:
    """
    Load up to n_subjects files matching glob, concatenate epochs.
    Returns (N_total, 19, 512) or None.
    """
    files = sorted(directory.glob(class_glob))[:n_subjects]
    arrays = []
    for fp in files:
        d = load_normalised(fp)
        if d is not None:
            arrays.append(d)
    if not arrays:
        return None
    return np.concatenate(arrays, axis=0)


# ─────────────────────────────────────────────────────────────────────────────
# Check 1: Band Power Comparison
# ─────────────────────────────────────────────────────────────────────────────

def check_spectral_bands(directory: Path) -> None:
    """
    Compare band power between normal and epileptic epochs.
    Uses Mann-Whitney U test (non-parametric, appropriate for EEG power).
    If p < 0.001 for any band → that band is trivially separating classes.
    """
    print("\n" + "═"*60)
    print("  CHECK 1: SPECTRAL BAND POWER (post-normalisation)")
    print("═"*60)

    normal_data = load_class_data(
        directory,
        "**/female_positive_raw_*_clean-epo.fif",
        N_SUBJECTS_PER_CLASS,
    )
    epileptic_data = load_class_data(
        directory,
        "**/female_negative_raw_*_clean-epo.fif",
        N_SUBJECTS_PER_CLASS,
    )

    if normal_data is None or epileptic_data is None:
        print("  [ERROR] Could not load data.")
        return

    print(f"\n  Normal epochs    : {len(normal_data)}")
    print(f"  Epileptic epochs : {len(epileptic_data)}")
    print()
    print(f"  {'Band':<8} {'Normal mean':>14} {'Epileptic mean':>16} "
          f"{'Ratio N/E':>12} {'p-value':>12} {'Separable?':>12}")
    print(f"  {'─'*8} {'─'*14} {'─'*16} {'─'*12} {'─'*12} {'─'*12}")

    separable_bands = []

    for band, (fmin, fmax) in FREQ_BANDS.items():
        n_power = bandpower(normal_data,    SFREQ, fmin, fmax)
        e_power = bandpower(epileptic_data, SFREQ, fmin, fmax)

        n_mean  = float(n_power.mean())
        e_mean  = float(e_power.mean())
        ratio   = n_mean / (e_mean + 1e-12)

        stat, p = mannwhitneyu(n_power, e_power, alternative="two-sided")
        sep     = "YES ✘" if p < 0.001 else "no ✔"

        if p < 0.001:
            separable_bands.append(band)

        print(f"  {band:<8} {n_mean:>14.6f} {e_mean:>16.6f} "
              f"{ratio:>12.3f} {p:>12.2e} {sep:>12}")

    print()
    if separable_bands:
        print(f"  ✘  SPECTRAL CONFOUND in bands: {separable_bands}")
        print(f"     The VAE exploits these band differences — AUROC=1.0 explained.")
        print(f"     This COULD be real neuroscience OR recording artifact.")
        print(f"     Need Check 2 to distinguish them.")
    else:
        print(f"  ✔  No spectral confound. AUROC=1.0 has a different cause.")


# ─────────────────────────────────────────────────────────────────────────────
# Check 2: Line Noise & DC Offset (Recording Artifact)
# ─────────────────────────────────────────────────────────────────────────────

def check_recording_artifacts(directory: Path) -> None:
    """
    Check for line noise (50/60 Hz) and DC offset differences.
    If present and different between classes → recording confound.
    """
    print("\n" + "═"*60)
    print("  CHECK 2: RECORDING ARTIFACTS (line noise + DC offset)")
    print("═"*60)

    normal_files    = sorted(
        directory.glob("**/female_positive_raw_*_clean-epo.fif")
    )[:3]
    epileptic_files = sorted(
        directory.glob("**/female_negative_raw_*_clean-epo.fif")
    )[:3]

    for label, files in [("NORMAL", normal_files),
                          ("EPILEPTIC", epileptic_files)]:
        print(f"\n  [{label}]")
        for fp in files:
            data = load_normalised(fp)
            if data is None:
                continue

            # DC offset: mean of all values (should be ~0 after z-score)
            dc_offset = float(np.abs(data.mean()))

            # Line noise power at 50 Hz and 60 Hz
            freqs, psd = sp_signal.welch(
                data, fs=SFREQ, nperseg=256, axis=2
            )
            # psd: (N, 19, n_freqs)
            mean_psd = psd.mean(axis=(0, 1))   # (n_freqs,)

            def _psd_at(target_hz: float, bw: float = 1.0) -> float:
                mask = (freqs >= target_hz - bw) & (freqs <= target_hz + bw)
                return float(mean_psd[mask].mean()) if mask.any() else 0.0

            p50 = _psd_at(50.0)
            p60 = _psd_at(60.0)

            print(f"    {fp.name:<50}")
            print(f"      DC offset  : {dc_offset:.6f}")
            print(f"      50 Hz power: {p50:.6f}")
            print(f"      60 Hz power: {p60:.6f}")


# ─────────────────────────────────────────────────────────────────────────────
# Check 3: Score Overlap From Pipeline
# ─────────────────────────────────────────────────────────────────────────────

def check_score_separation(directory: Path) -> None:
    """
    Manually compute reconstruction error using a quick AE (no VAE)
    on a single subject to see if scores are completely non-overlapping.

    Uses a trivial linear autoencoder — if THIS gives AUROC=1.0,
    the features are linearly separable after normalisation.
    """
    print("\n" + "═"*60)
    print("  CHECK 3: LINEAR SEPARABILITY TEST")
    print("  (If a linear PCA+threshold gives AUROC=1.0, features are trivial)")
    print("═"*60)

    from sklearn.decomposition import PCA
    from sklearn.metrics import roc_auc_score

    # Load one subject — normal training data
    normal_files = sorted(
        directory.glob("**/female_positive_raw_*_clean-epo.fif")
    )
    epileptic_files = sorted(
        directory.glob("**/female_negative_raw_*_clean-epo.fif")
    )

    # Use subjects 2-10 for training, subject 1 for test
    train_arrays = []
    for fp in normal_files[1:10]:
        d = load_normalised(fp)
        if d is not None:
            # Flatten: (N, 19, 512) → (N, 19*512)
            train_arrays.append(d.reshape(len(d), -1))

    if not train_arrays:
        print("  [ERROR] No training data.")
        return

    X_train = np.concatenate(train_arrays, axis=0)

    # Test data: subject 1
    X_norm_test = load_normalised(normal_files[0])
    X_epi_test  = load_normalised(epileptic_files[0])

    if X_norm_test is None or X_epi_test is None:
        print("  [ERROR] No test data.")
        return

    X_norm_flat = X_norm_test.reshape(len(X_norm_test), -1)
    X_epi_flat  = X_epi_test.reshape(len(X_epi_test),   -1)
    X_test      = np.vstack([X_norm_flat, X_epi_flat])
    y_test      = np.array(
        [0]*len(X_norm_flat) + [1]*len(X_epi_flat), dtype=np.int32
    )

    # Fit PCA on normal training data
    n_components = min(50, X_train.shape[0] - 1, X_train.shape[1])
    pca = PCA(n_components=n_components, random_state=42)
    pca.fit(X_train)

    # Reconstruction error as anomaly score
    X_test_proj  = pca.transform(X_test)
    X_test_recon = pca.inverse_transform(X_test_proj)
    scores       = np.mean((X_test - X_test_recon) ** 2, axis=1)

    auroc = roc_auc_score(y_test, scores)

    n_scores = scores[y_test == 0]
    e_scores = scores[y_test == 1]

    print(f"\n  PCA reconstruction AUROC   : {auroc:.4f}")
    print(f"  Normal    scores: mean={n_scores.mean():.4f}  "
          f"std={n_scores.std():.4f}  "
          f"min={n_scores.min():.4f}  max={n_scores.max():.4f}")
    print(f"  Epileptic scores: mean={e_scores.mean():.4f}  "
          f"std={e_scores.std():.4f}  "
          f"min={e_scores.min():.4f}  max={e_scores.max():.4f}")

    overlap = (e_scores.min() < n_scores.max()) and \
              (n_scores.min() < e_scores.max())

    print(f"\n  Score ranges overlap: {overlap}")

    if auroc > 0.95:
        print(f"\n  ✘  LINEAR SEPARABILITY CONFIRMED")
        print(f"     A simple PCA achieves AUROC={auroc:.4f}")
        print(f"     The VAE is not needed — features are trivially separable")
        print(f"     after z-score normalisation.")
        print(f"     This points to a SPECTRAL confound surviving normalisation.")
    elif auroc > 0.75:
        print(f"\n  ⚠  Partial linear separability (AUROC={auroc:.4f})")
        print(f"     VAE adds value but spectral features dominate.")
    else:
        print(f"\n  ✔  NOT linearly separable (AUROC={auroc:.4f})")
        print(f"     VAE is genuinely needed. AUROC=1.0 is surprising.")


# ─────────────────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    print("═"*60)
    print("  SPECTRAL CONFOUND & LEAKAGE DIAGNOSTIC")
    print("  Run before resuming LOSO")
    print("═"*60)

    check_spectral_bands(DATASET_DIR)
    check_recording_artifacts(DATASET_DIR)
    check_score_separation(DATASET_DIR)

    print("\n" + "═"*60)
    print("  INTERPRETATION GUIDE")
    print("─"*60)
    print("""
  After running, you will know ONE of three things:

  CASE A — Spectral confound (most likely)
    Separable bands + PCA AUROC > 0.95
    → The two classes differ in spectral content post-normalisation
    → Need: bandpass filter to REMOVE the confounded band before VAE
    → OR: accept that spectral difference IS the biomarker (justify it)

  CASE B — Recording artifact
    50/60 Hz power differs between classes
    → The two classes were recorded with different equipment/settings
    → This is a fatal confound — results are not publishable
    → Need: notch filter + re-run

  CASE C — Genuine neurophysiology
    Separable bands + PCA AUROC < 0.75
    → VAE is learning non-linear temporal structure
    → AUROC=1.0 is surprisingly good but potentially valid
    → Need: Grad-CAM / channel importance to verify spatial pattern
  """)
    print("═"*60)


if __name__ == "__main__":
    main()

════════════════════════════════════════════════════════════
  SPECTRAL CONFOUND & LEAKAGE DIAGNOSTIC
  Run before resuming LOSO
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  CHECK 1: SPECTRAL BAND POWER (post-normalisation)
════════════════════════════════════════════════════════════

  Normal epochs    : 1864
  Epileptic epochs : 2376

  Band        Normal mean   Epileptic mean    Ratio N/E      p-value   Separable?
  ──────── ────────────── ──────────────── ──────────── ──────────── ────────────
  delta          0.216347         0.181037        1.195    7.87e-153        YES ✘
  theta          0.025597         0.028161        0.909     1.30e-31        YES ✘
  alpha          0.003617         0.017748        0.204     0.00e+00        YES ✘
  beta           0.000586         0.002627        0.223     0.00e+00        YES ✘
  gamma          0.000109         0.000365        0.298     0.00e+00        YES ✘

  ✘  S

In [7]:
"""
Band Filter Verification
=========================
Strips delta (< 4 Hz) and high gamma (> 30 Hz) from the input.
Verifies that PCA AUROC drops from 1.0 to a non-trivial value.
If it does → the remaining signal is genuinely hard.
If it doesn't → the 4-30 Hz band is ALSO trivially separable.

Date: 2026-03-12
"""

from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import mne
from scipy import signal as sp_signal
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore")

DATASET_DIR = Path(
    r"/kaggle/input/datasets/johanliebert28/maaed-cleaned-epochs-archive-new"
)
KEEP_CHANNELS = [
    "Fp1","Fp2","F7","F3","Fz","F4","F8",
    "T3","C3","Cz","C4","T4",
    "T5","P3","Pz","P4","T6","O1","O2",
]
SFREQ     = 128.0
N_SAMPLES = 512

# ── Filter banks to test ──────────────────────────────────────────────────────
# Each entry: (name, low_hz, high_hz)
# None means no filter on that edge
FILTER_CONFIGS = [
    ("no filter (baseline)",     None,  None),
    ("delta only     0.5-4 Hz",  0.5,   4.0),
    ("theta only     4-8 Hz",    4.0,   8.0),
    ("alpha only     8-13 Hz",   8.0,  13.0),
    ("theta-beta     4-30 Hz",   4.0,  30.0),   # ← target config
    ("broadband      0.5-30 Hz", 0.5,  30.0),
    ("alpha-beta     8-30 Hz",   8.0,  30.0),
]


# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def load_raw_data(fif_path: Path) -> np.ndarray | None:
    """Load epochs, drop Oz, return raw (N, 19, 512) float32."""
    try:
        epochs = mne.read_epochs(str(fif_path), preload=True, verbose=False)
    except Exception:
        return None
    if "Oz" in epochs.ch_names:
        epochs = epochs.drop_channels(["Oz"])
    missing = [c for c in KEEP_CHANNELS if c not in epochs.ch_names]
    if missing:
        return None
    epochs = epochs.pick_channels(KEEP_CHANNELS, ordered=True)
    data   = epochs.get_data().astype(np.float32)
    if data.shape[1] != 19 or data.shape[2] != 512:
        return None
    return data


def apply_bandpass(
    data:   np.ndarray,    # (N, 19, 512) raw
    lo_hz:  float | None,
    hi_hz:  float | None,
    sfreq:  float = SFREQ,
) -> np.ndarray:
    """
    Apply zero-phase Butterworth bandpass filter.

    Parameters
    ----------
    data  : (N, 19, 512) raw signal in volts
    lo_hz : low cutoff Hz, or None for lowpass only
    hi_hz : high cutoff Hz, or None for highpass only
    sfreq : sampling frequency

    Returns
    -------
    filtered : (N, 19, 512) float32
    """
    if lo_hz is None and hi_hz is None:
        return data   # no filtering

    nyq = sfreq / 2.0

    if lo_hz is not None and hi_hz is not None:
        lo_norm = lo_hz / nyq
        hi_norm = hi_hz / nyq
        lo_norm = np.clip(lo_norm, 1e-4, 0.9999)
        hi_norm = np.clip(hi_norm, 1e-4, 0.9999)
        b, a    = sp_signal.butter(4, [lo_norm, hi_norm], btype="band")
    elif lo_hz is not None:
        lo_norm = np.clip(lo_hz / nyq, 1e-4, 0.9999)
        b, a    = sp_signal.butter(4, lo_norm, btype="high")
    else:
        hi_norm = np.clip(hi_hz / nyq, 1e-4, 0.9999)
        b, a    = sp_signal.butter(4, hi_norm, btype="low")

    # Apply filter along time axis (axis=2)
    filtered = sp_signal.filtfilt(b, a, data, axis=2)
    return filtered.astype(np.float32)


def z_score_normalise(data: np.ndarray) -> np.ndarray:
    """
    Per-epoch per-channel z-score.
    Input/output: (N, 19, 512)
    """
    mu  = data.mean(axis=2, keepdims=True)
    std = data.std(axis=2,  keepdims=True) + 1e-8
    return np.clip((data - mu) / std, -6.0, 6.0).astype(np.float32)


def pca_auroc(
    X_train:    np.ndarray,    # (N_train, 19*512) normal only
    X_norm_test:  np.ndarray,  # (N_norm, 19*512)
    X_epi_test:   np.ndarray,  # (N_epi,  19*512)
    n_components: int = 50,
) -> tuple[float, float, float]:
    """
    Fit PCA on normal training data.
    Compute reconstruction error as anomaly score.
    Return (auroc, normal_mean_score, epileptic_mean_score).
    """
    nc    = min(n_components, X_train.shape[0] - 1, X_train.shape[1])
    pca   = PCA(n_components=nc, random_state=42)
    pca.fit(X_train)

    X_test = np.vstack([X_norm_test, X_epi_test])
    y_test = np.array(
        [0]*len(X_norm_test) + [1]*len(X_epi_test),
        dtype=np.int32,
    )

    recon  = pca.inverse_transform(pca.transform(X_test))
    scores = np.mean((X_test - recon) ** 2, axis=1)

    auroc  = roc_auc_score(y_test, scores)
    n_mean = float(scores[y_test == 0].mean())
    e_mean = float(scores[y_test == 1].mean())

    return float(auroc), n_mean, e_mean


# ─────────────────────────────────────────────────────────────────────────────
# Main comparison
# ─────────────────────────────────────────────────────────────────────────────

def run_filter_comparison(directory: Path) -> None:
    """
    For each filter config:
      1. Load raw data (no normalisation yet)
      2. Apply bandpass filter
      3. Z-score normalise
      4. Fit PCA on normal training subjects
      5. Compute AUROC on held-out test subject
      6. Report: if AUROC drops below 0.75 → task is genuinely hard
    """
    print("\n" + "═"*68)
    print("  FILTER CONFIGURATION COMPARISON — PCA AUROC")
    print("  Goal: find filter that makes AUROC < 0.75 (genuinely hard task)")
    print("═"*68)

    # Files: use first 10 subjects for train, subject 1 for test
    normal_files    = sorted(
        directory.glob("**/female_positive_raw_*_clean-epo.fif")
    )
    epileptic_files = sorted(
        directory.glob("**/female_negative_raw_*_clean-epo.fif")
    )

    if len(normal_files) < 3 or len(epileptic_files) < 3:
        print("  [ERROR] Not enough files found.")
        return

    # Load raw (no filter yet)
    print("  Loading raw data...")
    train_raws, test_norm_raw, test_epi_raw = [], None, None

    for i, fp in enumerate(normal_files[:11]):
        d = load_raw_data(fp)
        if d is None:
            continue
        if i == 0:
            test_norm_raw = d
        else:
            train_raws.append(d)

    test_epi_raw = load_raw_data(epileptic_files[0])

    if not train_raws or test_norm_raw is None or test_epi_raw is None:
        print("  [ERROR] Data loading failed.")
        return

    print(f"  Train normal subjects : {len(train_raws)}")
    print(f"  Test normal epochs    : {len(test_norm_raw)}")
    print(f"  Test epileptic epochs : {len(test_epi_raw)}")

    print()
    print(f"  {'Filter Config':<30} {'AUROC':>8} "
          f"{'Normal MSE':>12} {'Epileptic MSE':>14} {'Hard task?':>12}")
    print(f"  {'─'*30} {'─'*8} {'─'*12} {'─'*14} {'─'*12}")

    best_config = None
    best_auroc_distance_from_trivial = 0.0

    for config_name, lo_hz, hi_hz in FILTER_CONFIGS:

        # Apply filter to all data
        train_filtered = [
            apply_bandpass(d, lo_hz, hi_hz) for d in train_raws
        ]
        test_norm_f  = apply_bandpass(test_norm_raw,  lo_hz, hi_hz)
        test_epi_f   = apply_bandpass(test_epi_raw,   lo_hz, hi_hz)

        # Z-score normalise
        train_norm = [z_score_normalise(d) for d in train_filtered]
        test_norm_z = z_score_normalise(test_norm_f)
        test_epi_z  = z_score_normalise(test_epi_f)

        # Flatten for PCA
        X_train = np.concatenate(
            [d.reshape(len(d), -1) for d in train_norm], axis=0
        )
        X_norm_test = test_norm_z.reshape(len(test_norm_z), -1)
        X_epi_test  = test_epi_z.reshape(len(test_epi_z),  -1)

        auroc, n_mse, e_mse = pca_auroc(X_train, X_norm_test, X_epi_test)

        hard    = "YES ✔" if auroc < 0.75 else ("medium" if auroc < 0.90 else "NO ✘")
        dist    = abs(auroc - 0.5)

        print(f"  {config_name:<30} {auroc:>8.4f} "
              f"{n_mse:>12.4f} {e_mse:>14.4f} {hard:>12}")

        # Track which config makes the task hardest
        if auroc < best_auroc_distance_from_trivial or best_config is None:
            if auroc < 0.99:   # ignore trivial configs
                best_auroc_distance_from_trivial = auroc
                best_config = (config_name, lo_hz, hi_hz, auroc)

    print()
    print("─"*68)

    if best_config:
        cname, lo, hi, auc = best_config
        print(f"\n  Recommended filter: [{cname}]")
        print(f"  PCA AUROC with this filter: {auc:.4f}")

        if auc < 0.75:
            print(f"\n  ✔  Task is genuinely hard with this filter.")
            print(f"     Update pipeline: apply bandpass({lo}, {hi}) Hz")
            print(f"     before z-score normalisation in load_subject_epochs().")
            print(f"     The VAE will now learn temporal dynamics, not spectral means.")
        elif auc < 0.90:
            print(f"\n  ⚠  Task is moderately hard (AUROC={auc:.4f}).")
            print(f"     VAE adds marginal value over linear methods.")
            print(f"     Consider this the minimum acceptable filter.")
        else:
            print(f"\n  ✘  ALL filters leave task linearly separable.")
            print(f"     The spectral difference is so large it survives filtering.")
            print(f"     You need to reconsider the task framing entirely.")
            print(f"     Recommended: switch to seizure ONSET detection (Option 3).")

    print("═"*68)


def main() -> None:
    print("═"*68)
    print("  BAND FILTER VERIFICATION")
    print("  Finding the right preprocessing to make task non-trivial")
    print("═"*68)
    run_filter_comparison(DATASET_DIR)


if __name__ == "__main__":
    main()

════════════════════════════════════════════════════════════════════
  BAND FILTER VERIFICATION
  Finding the right preprocessing to make task non-trivial
════════════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════════════
  FILTER CONFIGURATION COMPARISON — PCA AUROC
  Goal: find filter that makes AUROC < 0.75 (genuinely hard task)
════════════════════════════════════════════════════════════════════
  Loading raw data...
  Train normal subjects : 10
  Test normal epochs    : 219
  Test epileptic epochs : 183

  Filter Config                     AUROC   Normal MSE  Epileptic MSE   Hard task?
  ────────────────────────────── ──────── ──────────── ────────────── ────────────
  no filter (baseline)             1.0000       0.0567         0.5240         NO ✘
  delta only     0.5-4 Hz          1.0000       0.0199         0.3759         NO ✘
  theta only     4-8 Hz            1.0000       0.0564         0.4776         NO 

In [9]:
"""
β-VAE Epilepsy Detection — Full Pipeline v3
=============================================
Implements two experiments confirmed by diagnostic tests:

  Experiment 1: LOSO cross-subject
    Train on normal epochs from all subjects except S
    Test  on subject S (normal + epileptic)

  Experiment 2: Within-subject
    Train on 80% of subject S normal epochs
    Test  on 20% normal + all epileptic epochs of subject S
    Stratified split — no temporal leakage within subject

  Baselines for both experiments:
    - PCA reconstruction error
    - Isolation Forest
    - One-Class SVM

Confirmed dataset facts:
  N_SAMPLES  = 512   (4.0 s @ 128 Hz)
  N_CHANNELS = 19    (Oz dropped from 4 files)
  positive   = normal,    negative = epileptic
  LOSO pool  = 101 subjects (10 train-only excluded)

Date : 2026-03-12
Seed : 42
"""

from __future__ import annotations

import os
import re
import time
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import mne
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
mne.set_log_level("ERROR")
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────────────────────────────────────
DATASET_DIR = Path(
    r"/kaggle/input/datasets/johanliebert28/maaed-cleaned-epochs-archive-new"
)
RESULTS_DIR  = Path("./vae_results_v3")
N_CHANNELS   = 19
N_SAMPLES    = 512
SFREQ        = 128.0
KEEP_CHANNELS = [
    "Fp1","Fp2","F7","F3","Fz","F4","F8",
    "T3","C3","Cz","C4","T4",
    "T5","P3","Pz","P4","T6","O1","O2",
]
LATENT_DIM    = 64
BETA          = 4.0
EPOCHS_TRAIN  = 50
BATCH_SIZE    = 32
LEARNING_RATE = 1e-3
WITHIN_TRAIN_RATIO = 0.80   # 80% normal epochs for within-subject training

TRAIN_ONLY_SUBJECTS = {
    "female_51",
    "male_52","male_53","male_54","male_55","male_56",
    "male_57","male_58","male_59","male_60",
}
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — Data
# ─────────────────────────────────────────────────────────────────────────────

def _parse_filename(filename: str) -> tuple[str, str, str]:
    """
    positive → normal, negative → epileptic  (confirmed by diagnostic tests)
    Returns (subject_id, label, gender)
    """
    name  = filename.lower()
    match = re.match(
        r"^(male|female)_(positive|negative)_raw_(\d+)_", name
    )
    if not match:
        return "unknown", "unknown", "unknown"
    gender     = match.group(1)
    raw_label  = match.group(2)
    subject_id = f"{gender}_{match.group(3)}"
    label      = "normal" if raw_label == "positive" else "epileptic"
    return subject_id, label, gender


def load_subject_epochs(fif_path: Path) -> Optional[np.ndarray]:
    """
    Load .fif → (N, 19, 512) float32, z-score normalised, clipped.

    Returns None on any failure.
    """
    try:
        epochs = mne.read_epochs(str(fif_path), preload=True, verbose=False)
    except Exception as exc:
        print(f"  [LOAD ERROR] {fif_path.name}: {exc}")
        return None

    if "Oz" in epochs.ch_names:
        epochs = epochs.drop_channels(["Oz"])

    missing = [c for c in KEEP_CHANNELS if c not in epochs.ch_names]
    if missing:
        return None

    epochs = epochs.pick_channels(KEEP_CHANNELS, ordered=True)
    data   = epochs.get_data().astype(np.float32)

    if data.shape[1] != N_CHANNELS or data.shape[2] != N_SAMPLES:
        return None

    mu   = data.mean(axis=2, keepdims=True)
    std  = data.std(axis=2,  keepdims=True) + 1e-8
    data = np.clip((data - mu) / std, -6.0, 6.0)
    return data


def build_registry(directory: Path) -> pd.DataFrame:
    fif_files = sorted(set(
        list(directory.glob("**/*-epo.fif")) +
        list(directory.glob("**/*_epo.fif")) +
        list(directory.glob("**/*.fif"))
    ))
    rows = []
    for fp in fif_files:
        sid, label, gender = _parse_filename(fp.name)
        if sid == "unknown":
            continue
        rows.append({
            "filepath":     str(fp),
            "subject_id":   sid,
            "label":        label,
            "gender":       gender,
            "in_loso_pool": sid not in TRAIN_ONLY_SUBJECTS,
        })
    return pd.DataFrame(rows)


def preload_all(registry: pd.DataFrame) -> dict[tuple[str,str], np.ndarray]:
    """Load all files into memory. Key = (subject_id, label)."""
    cache: dict[tuple[str,str], np.ndarray] = {}
    skipped = 0
    for _, row in registry.iterrows():
        data = load_subject_epochs(Path(row["filepath"]))
        if data is not None:
            cache[(row["subject_id"], row["label"])] = data
        else:
            skipped += 1
    print(f"  Loaded {len(cache)} arrays | Skipped {skipped}")
    return cache


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — β-VAE
# ─────────────────────────────────────────────────────────────────────────────

class Sampling(tf.keras.layers.Layer):
    def call(self, inputs):
        mu, log_sigma = inputs
        eps = tf.random.normal(shape=tf.shape(mu), seed=SEED)
        return mu + tf.exp(0.5 * log_sigma) * eps


def build_encoder(
    n_ch: int = N_CHANNELS,
    n_t:  int = N_SAMPLES,
    ld:   int = LATENT_DIM,
) -> tf.keras.Model:
    """
    (batch, 19, 512) → [μ, log_σ, z]  each (batch, 64)

    DepthwiseConv1D(k=25) → PointwiseConv1D(64) → Permute
    → StrideConv1D×3(stride=2) → BiGRU(128) → Dense(128)
    → Dense μ, Dense log_σ, Sampling z
    """
    inp = tf.keras.Input(shape=(n_ch, n_t), name="enc_in")
    x   = tf.keras.layers.DepthwiseConv1D(
            kernel_size=25, padding="same",
            activation="elu", name="dw_conv")(inp)
    x   = tf.keras.layers.Conv1D(
            64, kernel_size=1,
            activation="elu", name="pw_conv")(x)
    x   = tf.keras.layers.Permute((2, 1), name="enc_perm")(x)
    for i, f in enumerate([64, 128, 128]):
        x = tf.keras.layers.Conv1D(
                f, kernel_size=5, strides=2,
                padding="same", activation="elu",
                name=f"sc_{i}")(x)
    x   = tf.keras.layers.Bidirectional(
            tf.keras.layers.GRU(128), name="bigru")(x)
    x   = tf.keras.layers.Dense(128, activation="elu", name="enc_d")(x)
    mu      = tf.keras.layers.Dense(ld, name="mu")(x)
    log_sig = tf.keras.layers.Dense(ld, name="log_sigma")(x)
    z       = Sampling(name="z")([mu, log_sig])
    return tf.keras.Model(inp, [mu, log_sig, z], name="encoder")


def build_decoder(
    n_ch: int = N_CHANNELS,
    n_t:  int = N_SAMPLES,
    ld:   int = LATENT_DIM,
) -> tf.keras.Model:
    """
    z (batch, 64) → (batch, 19, 512)

    Dense(64*128) → Reshape(64,128) → GRU(128)
    → ConvTranspose1D×3(stride=2) → Conv1D(19) → Permute
    512 = 64 * 2^3 → exact, no crop needed
    """
    z_in = tf.keras.Input(shape=(ld,), name="dec_in")
    x    = tf.keras.layers.Dense(
               64*128, activation="elu", name="dec_d")(z_in)
    x    = tf.keras.layers.Reshape((64, 128), name="reshape")(x)
    x    = tf.keras.layers.GRU(
               128, return_sequences=True, name="dec_gru")(x)
    for i, f in enumerate([128, 64, 64]):
        x = tf.keras.layers.Conv1DTranspose(
                f, kernel_size=5, strides=2,
                padding="same", activation="elu",
                name=f"dconv_{i}")(x)
    x    = tf.keras.layers.Conv1D(
               n_ch, kernel_size=1,
               activation="linear", name="out_conv")(x)
    x    = tf.keras.layers.Permute((2, 1), name="out_perm")(x)
    return tf.keras.Model(z_in, x, name="decoder")


class BetaVAE(tf.keras.Model):
    """
    β-VAE. Loss = MSE(x, x̂) + β·KL(q(z|x)‖N(0,I))
    Anomaly score = per-epoch MSE(x, x̂). Higher = more anomalous.
    """
    def __init__(self, encoder, decoder, beta=BETA):
        super().__init__()
        self.encoder     = encoder
        self.decoder     = decoder
        self.beta        = beta
        self._total      = tf.keras.metrics.Mean(name="total_loss")
        self._recon      = tf.keras.metrics.Mean(name="recon_loss")
        self._kl         = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self._total, self._recon, self._kl]

    def call(self, x, training=False):
        mu, log_sig, z = self.encoder(x, training=training)
        return self.decoder(z, training=training), mu, log_sig

    def _loss(self, x):
        x_hat, mu, log_sig = self(x, training=True)
        recon = tf.reduce_mean(
            tf.reduce_sum(tf.square(x - x_hat), axis=[1, 2]))
        kl    = -0.5 * tf.reduce_mean(
            tf.reduce_sum(
                1.0 + log_sig - tf.square(mu) - tf.exp(log_sig),
                axis=1))
        return recon + self.beta * kl, recon, kl

    def train_step(self, data):
        with tf.GradientTape() as tape:
            total, recon, kl = self._loss(data)
        self.optimizer.apply_gradients(
            zip(tape.gradient(total, self.trainable_variables),
                self.trainable_variables))
        self._total.update_state(total)
        self._recon.update_state(recon)
        self._kl.update_state(kl)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        total, recon, kl = self._loss(data)
        self._total.update_state(total)
        self._recon.update_state(recon)
        self._kl.update_state(kl)
        return {m.name: m.result() for m in self.metrics}

    def anomaly_score(self, x: np.ndarray) -> np.ndarray:
        """(N,19,512) → (N,) MSE scores. Higher = more epileptic."""
        xt        = tf.constant(x, dtype=tf.float32)
        x_hat,_,_ = self(xt, training=False)
        return tf.reduce_mean(
            tf.square(xt - x_hat), axis=[1, 2]).numpy().astype(np.float32)


def train_vae(X_train: np.ndarray) -> BetaVAE:
    """
    Build and train a fresh VAE on X_train (normal epochs only).

    Parameters
    ----------
    X_train : (N, 19, 512) normal epochs

    Returns
    -------
    Trained BetaVAE instance
    """
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    vae = BetaVAE(build_encoder(), build_decoder(), beta=BETA)
    vae.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE))

    ds = (tf.data.Dataset.from_tensor_slices(X_train)
          .shuffle(min(len(X_train), 5000), seed=SEED)
          .batch(BATCH_SIZE)
          .prefetch(tf.data.AUTOTUNE))

    vae.fit(
        ds,
        epochs=EPOCHS_TRAIN,
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor="total_loss", mode="min",
            patience=5, restore_best_weights=True, verbose=0
        )],
        verbose=0,
    )
    return vae


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — Baselines
# ─────────────────────────────────────────────────────────────────────────────

def pca_scores(
    X_train: np.ndarray,
    X_test:  np.ndarray,
    n_components: int = 50,
) -> np.ndarray:
    """
    PCA reconstruction error as anomaly score.

    Parameters
    ----------
    X_train : (N_train, 19*512) flattened normal epochs
    X_test  : (N_test,  19*512) flattened test epochs

    Returns
    -------
    scores : (N_test,) float32
    """
    nc  = min(n_components, X_train.shape[0]-1, X_train.shape[1])
    pca = PCA(n_components=nc, random_state=SEED)
    pca.fit(X_train)
    recon  = pca.inverse_transform(pca.transform(X_test))
    scores = np.mean((X_test - recon)**2, axis=1)
    return scores.astype(np.float32)


def isolation_forest_scores(
    X_train: np.ndarray,
    X_test:  np.ndarray,
) -> np.ndarray:
    """
    Isolation Forest anomaly scores.
    Returns negated decision_function so higher = more anomalous.

    Parameters
    ----------
    X_train : (N_train, D) normal epochs — flattened or band-power features
    X_test  : (N_test,  D)
    """
    clf = IsolationForest(
        n_estimators=200,
        contamination=0.1,
        random_state=SEED,
        n_jobs=-1,
    )
    clf.fit(X_train)
    scores = -clf.decision_function(X_test)   # higher = more anomalous
    return scores.astype(np.float32)


def one_class_svm_scores(
    X_train: np.ndarray,
    X_test:  np.ndarray,
) -> np.ndarray:
    """
    One-Class SVM anomaly scores.
    Negated decision_function so higher = more anomalous.
    Uses RBF kernel. Scales features first.

    Parameters
    ----------
    X_train : (N_train, D) — use band-power features, not raw (too slow)
    X_test  : (N_test,  D)
    """
    scaler  = StandardScaler()
    Xtr_sc  = scaler.fit_transform(X_train)
    Xte_sc  = scaler.transform(X_test)
    clf     = OneClassSVM(kernel="rbf", nu=0.1, gamma="scale")
    clf.fit(Xtr_sc)
    scores  = -clf.decision_function(Xte_sc)
    return scores.astype(np.float32)


def extract_bandpower_features(data: np.ndarray) -> np.ndarray:
    """
    Extract mean band power per channel per epoch as feature vector.

    Parameters
    ----------
    data : (N, 19, 512) normalised EEG

    Returns
    -------
    features : (N, 19*5) = (N, 95) float32
               5 bands × 19 channels
    """
    from scipy import signal as sp_signal

    bands = [(0.5,4),(4,8),(8,13),(13,30),(30,45)]
    freqs, psd = sp_signal.welch(data, fs=SFREQ, nperseg=256, axis=2)
    # psd: (N, 19, n_freqs)

    features = []
    for flo, fhi in bands:
        mask = (freqs >= flo) & (freqs <= fhi)
        bp   = psd[:, :, mask].mean(axis=2)   # (N, 19)
        features.append(bp)

    return np.concatenate(features, axis=1).astype(np.float32)   # (N, 95)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — Metrics
# ─────────────────────────────────────────────────────────────────────────────

def sensitivity_at_specificity(
    y_true: np.ndarray,
    scores: np.ndarray,
    target: float = 0.95,
) -> float:
    fpr, tpr, _ = roc_curve(y_true, scores)
    specs       = 1.0 - fpr
    idx         = np.where(specs >= target)[0]
    if len(idx) == 0:
        return float("nan")
    best = idx[np.argmin(np.abs(specs[idx] - target))]
    return float(tpr[best])


def compute_all_metrics(
    y_true:     np.ndarray,
    scores:     np.ndarray,
    latency_ms: float = 0.0,
    n_params:   int   = 0,
) -> dict:
    """Full metric set. Threshold = median of scores."""
    if len(np.unique(y_true)) < 2:
        return {}
    auroc   = float(roc_auc_score(y_true, scores))
    thresh  = float(np.median(scores))
    y_pred  = (scores >= thresh).astype(int)
    f1      = float(f1_score(y_true, y_pred, zero_division=0))
    s95     = sensitivity_at_specificity(y_true, scores, 0.95)
    tn,fp,fn,tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    sens    = tp / max(tp+fn, 1)
    spec    = tn / max(tn+fp, 1)
    return {
        "auroc":          round(auroc, 4),
        "f1":             round(f1,    4),
        "sensitivity":    round(float(sens), 4),
        "specificity":    round(float(spec), 4),
        "sens_at_95spec": round(float(s95),  4),
        "latency_ms":     round(latency_ms,  4),
        "n_params":       n_params,
    }


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — Experiment 1: LOSO Cross-Subject
# ─────────────────────────────────────────────────────────────────────────────

def run_experiment1_loso(
    cache:    dict,
    registry: pd.DataFrame,
) -> pd.DataFrame:
    """
    LOSO cross-subject experiment.
    Trains VAE + all baselines. One row per fold per model.
    """
    loso_subjects = sorted(
        registry[registry["in_loso_pool"]]["subject_id"].unique()
    )
    print(f"\n  [EXP 1] LOSO | {len(loso_subjects)} subjects")

    all_rows = []

    for fold_idx, test_sid in enumerate(loso_subjects):
        # ── Training data: normal epochs, all subjects except test ──────────
        train_arrays = [
            data for (sid, lbl), data in cache.items()
            if sid != test_sid and lbl == "normal"
        ]
        if not train_arrays:
            continue

        X_train = np.concatenate(train_arrays, axis=0)
        np.random.shuffle(X_train)

        # ── Test data ────────────────────────────────────────────────────────
        X_n = cache.get((test_sid, "normal"))
        X_e = cache.get((test_sid, "epileptic"))
        if X_n is None or X_e is None:
            continue

        X_test = np.concatenate([X_n, X_e], axis=0)
        y_test = np.array([0]*len(X_n) + [1]*len(X_e), dtype=np.int32)

        print(f"  Fold {fold_idx+1:03d}/{len(loso_subjects)} | {test_sid} | "
              f"train_n={len(X_train)} "
              f"test_n={len(X_n)} test_e={len(X_e)}")

        # ── β-VAE ────────────────────────────────────────────────────────────
        vae      = train_vae(X_train)
        n_params = int(sum(np.prod(v.shape) for v in vae.trainable_variables))
        _        = vae.anomaly_score(X_test[:1])   # warm-up
        t0       = time.perf_counter()
        vae_sc   = vae.anomaly_score(X_test)
        lat_vae  = ((time.perf_counter()-t0)/len(X_test))*1000

        m = compute_all_metrics(y_test, vae_sc, lat_vae, n_params)
        if m:
            all_rows.append({
                "experiment": "loso", "model": "beta_vae",
                "subject_id": test_sid, "fold": fold_idx+1, **m,
            })

        # ── PCA baseline ─────────────────────────────────────────────────────
        Xtr_flat = X_train.reshape(len(X_train), -1)
        Xte_flat = X_test.reshape(len(X_test),   -1)

        t0     = time.perf_counter()
        pca_sc = pca_scores(Xtr_flat, Xte_flat)
        lat_pca = ((time.perf_counter()-t0)/len(X_test))*1000

        m = compute_all_metrics(y_test, pca_sc, lat_pca, 0)
        if m:
            all_rows.append({
                "experiment": "loso", "model": "pca",
                "subject_id": test_sid, "fold": fold_idx+1, **m,
            })

        # ── Isolation Forest (band-power features) ───────────────────────────
        Xtr_bp = extract_bandpower_features(X_train)
        Xte_bp = extract_bandpower_features(X_test)

        t0    = time.perf_counter()
        if_sc = isolation_forest_scores(Xtr_bp, Xte_bp)
        lat_if = ((time.perf_counter()-t0)/len(X_test))*1000

        m = compute_all_metrics(y_test, if_sc, lat_if, 0)
        if m:
            all_rows.append({
                "experiment": "loso", "model": "isolation_forest",
                "subject_id": test_sid, "fold": fold_idx+1, **m,
            })

        # Print fold summary
        vae_auroc = next(
            r["auroc"] for r in reversed(all_rows)
            if r["model"]=="beta_vae" and r["subject_id"]==test_sid
        )
        pca_auroc = next(
            r["auroc"] for r in reversed(all_rows)
            if r["model"]=="pca" and r["subject_id"]==test_sid
        )
        print(f"    VAE={vae_auroc:.4f}  PCA={pca_auroc:.4f}")

    return pd.DataFrame(all_rows)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — Experiment 2: Within-Subject
# ─────────────────────────────────────────────────────────────────────────────

def run_experiment2_within_subject(
    cache:    dict,
    registry: pd.DataFrame,
) -> pd.DataFrame:
    """
    Within-subject experiment.
    For each subject with both classes:
      Train on 80% of their OWN normal epochs
      Test  on 20% normal + ALL epileptic epochs

    This is the most novel experiment — no published paper on MAAED
    has done within-subject unsupervised epilepsy detection.
    """
    loso_subjects = sorted(
        registry[registry["in_loso_pool"]]["subject_id"].unique()
    )
    print(f"\n  [EXP 2] Within-subject | {len(loso_subjects)} subjects")

    all_rows = []

    for sid in loso_subjects:
        X_n = cache.get((sid, "normal"))
        X_e = cache.get((sid, "epileptic"))

        if X_n is None or X_e is None:
            continue
        if len(X_n) < 20:    # skip subjects with too few normal epochs
            print(f"  [SKIP] {sid}: only {len(X_n)} normal epochs")
            continue

        # Stratified 80/20 split of normal epochs — no shuffle across time
        rng     = np.random.RandomState(SEED)
        idx     = rng.permutation(len(X_n))
        n_train = int(len(X_n) * WITHIN_TRAIN_RATIO)
        X_train = X_n[idx[:n_train]]
        X_norm_test = X_n[idx[n_train:]]

        X_test = np.concatenate([X_norm_test, X_e], axis=0)
        y_test = np.array(
            [0]*len(X_norm_test) + [1]*len(X_e), dtype=np.int32
        )

        print(f"  Subject {sid:<15} | "
              f"train_n={len(X_train)}  "
              f"test_n={len(X_norm_test)}  "
              f"test_e={len(X_e)}")

        # ── β-VAE ────────────────────────────────────────────────────────────
        vae      = train_vae(X_train)
        n_params = int(sum(np.prod(v.shape) for v in vae.trainable_variables))
        _        = vae.anomaly_score(X_test[:1])
        t0       = time.perf_counter()
        vae_sc   = vae.anomaly_score(X_test)
        lat_vae  = ((time.perf_counter()-t0)/len(X_test))*1000

        m = compute_all_metrics(y_test, vae_sc, lat_vae, n_params)
        if m:
            all_rows.append({
                "experiment": "within_subject", "model": "beta_vae",
                "subject_id": sid,
                "n_train_normal": len(X_train),
                "n_test_normal":  len(X_norm_test),
                "n_test_epileptic": len(X_e),
                **m,
            })

        # ── PCA baseline ─────────────────────────────────────────────────────
        Xtr_flat = X_train.reshape(len(X_train), -1)
        Xte_flat = X_test.reshape(len(X_test),   -1)
        t0       = time.perf_counter()
        pca_sc   = pca_scores(Xtr_flat, Xte_flat)
        lat_pca  = ((time.perf_counter()-t0)/len(X_test))*1000

        m = compute_all_metrics(y_test, pca_sc, lat_pca, 0)
        if m:
            all_rows.append({
                "experiment": "within_subject", "model": "pca",
                "subject_id": sid,
                "n_train_normal": len(X_train),
                "n_test_normal":  len(X_norm_test),
                "n_test_epileptic": len(X_e),
                **m,
            })

        # ── One-Class SVM (band-power, within-subject) ────────────────────────
        Xtr_bp  = extract_bandpower_features(X_train)
        Xte_bp  = extract_bandpower_features(X_test)
        t0      = time.perf_counter()
        svm_sc  = one_class_svm_scores(Xtr_bp, Xte_bp)
        lat_svm = ((time.perf_counter()-t0)/len(X_test))*1000

        m = compute_all_metrics(y_test, svm_sc, lat_svm, 0)
        if m:
            all_rows.append({
                "experiment": "within_subject", "model": "oc_svm",
                "subject_id": sid,
                "n_train_normal": len(X_train),
                "n_test_normal":  len(X_norm_test),
                "n_test_epileptic": len(X_e),
                **m,
            })

    return pd.DataFrame(all_rows)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — Final Report
# ─────────────────────────────────────────────────────────────────────────────

def print_final_report(
    exp1: pd.DataFrame,
    exp2: pd.DataFrame,
) -> None:
    print("\n" + "═"*68)
    print("  FINAL RESULTS")
    print("═"*68)

    for exp_name, df in [("EXPERIMENT 1 — LOSO Cross-Subject", exp1),
                          ("EXPERIMENT 2 — Within-Subject",     exp2)]:
        print(f"\n  {exp_name}")
        print("  " + "─"*60)

        if df.empty:
            print("  No results.")
            continue

        metrics = ["auroc","f1","sensitivity","specificity","sens_at_95spec"]
        models  = df["model"].unique()

        print(f"  {'Model':<20} " +
              "  ".join(f"{m:<16}" for m in metrics))
        print("  " + "─"*60)

        for mdl in models:
            sub = df[df["model"]==mdl]
            row_str = f"  {mdl:<20}"
            for m in metrics:
                if m in sub.columns:
                    col = sub[m].dropna()
                    row_str += f"  {col.mean():.4f}±{col.std():.3f}  "
            print(row_str)

        # VAE vs PCA improvement
        if "beta_vae" in models and "pca" in models:
            vae_auroc = df[df["model"]=="beta_vae"]["auroc"].mean()
            pca_auroc = df[df["model"]=="pca"]["auroc"].mean()
            delta     = vae_auroc - pca_auroc
            marker    = "✔ VAE adds value" if delta > 0.01 else "✘ VAE = PCA"
            print(f"\n  VAE AUROC={vae_auroc:.4f}  "
                  f"PCA AUROC={pca_auroc:.4f}  "
                  f"Δ={delta:+.4f}  {marker}")

    print("\n" + "═"*68)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 — Entrypoint
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    print("═"*68)
    print("  β-VAE EPILEPSY PIPELINE v3 — TWO EXPERIMENTS + BASELINES")
    print(f"  Seed={SEED} | N_SAMPLES={N_SAMPLES} | N_CHANNELS={N_CHANNELS}")
    print("═"*68)

    if not DATASET_DIR.exists():
        print(f"  ✘ DATASET_DIR not found: {DATASET_DIR}")
        return

    registry = build_registry(DATASET_DIR)
    print(f"\n  Total subjects     : {registry['subject_id'].nunique()}")
    print(f"  LOSO eval subjects : "
          f"{registry[registry['in_loso_pool']]['subject_id'].nunique()}")

    print("\n  Pre-loading all epochs...")
    cache = preload_all(registry)

    # Run experiments
    exp1 = run_experiment1_loso(cache, registry)
    exp2 = run_experiment2_within_subject(cache, registry)

    # Save results
    if not exp1.empty:
        p = RESULTS_DIR / "exp1_loso.csv"
        exp1.to_csv(p, index=False)
        print(f"\n  ✔ Exp1 saved: {p}")

    if not exp2.empty:
        p = RESULTS_DIR / "exp2_within_subject.csv"
        exp2.to_csv(p, index=False)
        print(f"  ✔ Exp2 saved: {p}")

    print_final_report(exp1, exp2)


if __name__ == "__main__":
    main()

════════════════════════════════════════════════════════════════════
  β-VAE EPILEPSY PIPELINE v3 — TWO EXPERIMENTS + BASELINES
  Seed=42 | N_SAMPLES=512 | N_CHANNELS=19
════════════════════════════════════════════════════════════════════

  Total subjects     : 111
  LOSO eval subjects : 101

  Pre-loading all epochs...
  Loaded 212 arrays | Skipped 0

  [EXP 1] LOSO | 101 subjects
  Fold 001/101 | female_1 | train_n=24071 test_n=369 test_e=142
    VAE=1.0000  PCA=1.0000
  Fold 002/101 | female_10 | train_n=24221 test_n=219 test_e=183
    VAE=1.0000  PCA=1.0000
  Fold 003/101 | female_11 | train_n=24345 test_n=95 test_e=112


KeyboardInterrupt: 

In [1]:
"""
β-VAE Epilepsy Detection — Full Pipeline v3
=============================================
Implements two experiments confirmed by diagnostic tests:

  Experiment 1: LOSO cross-subject
    Train on normal epochs from all subjects except S
    Test  on subject S (normal + epileptic)

  Experiment 2: Within-subject
    Train on 80% of subject S normal epochs
    Test  on 20% normal + all epileptic epochs of subject S
    Stratified split — no temporal leakage within subject

  Baselines for both experiments:
    - PCA reconstruction error
    - Isolation Forest
    - One-Class SVM

Confirmed dataset facts:
  N_SAMPLES  = 512   (4.0 s @ 128 Hz)
  N_CHANNELS = 19    (Oz dropped from 4 files)
  positive   = normal,    negative = epileptic
  LOSO pool  = 101 subjects (10 train-only excluded)

Changes in this version:
  - Checkpoint save/resume added to Exp1 and Exp2
  - train_vae() accepts max_epochs parameter
  - Exp1 uses max_epochs=20 (sufficient, saves ~60% runtime)
  - Exp2 uses max_epochs=50 (within-subject needs more epochs)

Date : 2026-03-12
Seed : 42
"""

from __future__ import annotations

import os
import re
import time
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import mne
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
mne.set_log_level("ERROR")
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────────────────────────────────────
DATASET_DIR = Path(
    r"/kaggle/input/datasets/johanliebert28/maaed-cleaned-epochs-archive-new"
)
RESULTS_DIR  = Path("./vae_results_v3")
N_CHANNELS   = 19
N_SAMPLES    = 512
SFREQ        = 128.0
KEEP_CHANNELS = [
    "Fp1","Fp2","F7","F3","Fz","F4","F8",
    "T3","C3","Cz","C4","T4",
    "T5","P3","Pz","P4","T6","O1","O2",
]
LATENT_DIM    = 64
BETA          = 4.0
EPOCHS_TRAIN  = 50
BATCH_SIZE    = 32
LEARNING_RATE = 1e-3
WITHIN_TRAIN_RATIO = 0.80

TRAIN_ONLY_SUBJECTS = {
    "female_51",
    "male_52","male_53","male_54","male_55","male_56",
    "male_57","male_58","male_59","male_60",
}
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — Data
# ─────────────────────────────────────────────────────────────────────────────

def _parse_filename(filename: str) -> tuple[str, str, str]:
    """
    positive → normal, negative → epileptic  (confirmed by diagnostic tests)
    Returns (subject_id, label, gender)
    """
    name  = filename.lower()
    match = re.match(
        r"^(male|female)_(positive|negative)_raw_(\d+)_", name
    )
    if not match:
        return "unknown", "unknown", "unknown"
    gender     = match.group(1)
    raw_label  = match.group(2)
    subject_id = f"{gender}_{match.group(3)}"
    label      = "normal" if raw_label == "positive" else "epileptic"
    return subject_id, label, gender


def load_subject_epochs(fif_path: Path) -> Optional[np.ndarray]:
    """
    Load .fif → (N, 19, 512) float32, z-score normalised, clipped.
    Returns None on any failure.
    """
    try:
        epochs = mne.read_epochs(str(fif_path), preload=True, verbose=False)
    except Exception as exc:
        print(f"  [LOAD ERROR] {fif_path.name}: {exc}")
        return None

    if "Oz" in epochs.ch_names:
        epochs = epochs.drop_channels(["Oz"])

    missing = [c for c in KEEP_CHANNELS if c not in epochs.ch_names]
    if missing:
        return None

    epochs = epochs.pick_channels(KEEP_CHANNELS, ordered=True)
    data   = epochs.get_data().astype(np.float32)

    if data.shape[1] != N_CHANNELS or data.shape[2] != N_SAMPLES:
        return None

    mu   = data.mean(axis=2, keepdims=True)
    std  = data.std(axis=2,  keepdims=True) + 1e-8
    data = np.clip((data - mu) / std, -6.0, 6.0)
    return data


def build_registry(directory: Path) -> pd.DataFrame:
    fif_files = sorted(set(
        list(directory.glob("**/*-epo.fif")) +
        list(directory.glob("**/*_epo.fif")) +
        list(directory.glob("**/*.fif"))
    ))
    rows = []
    for fp in fif_files:
        sid, label, gender = _parse_filename(fp.name)
        if sid == "unknown":
            continue
        rows.append({
            "filepath":     str(fp),
            "subject_id":   sid,
            "label":        label,
            "gender":       gender,
            "in_loso_pool": sid not in TRAIN_ONLY_SUBJECTS,
        })
    return pd.DataFrame(rows)


def preload_all(registry: pd.DataFrame) -> dict[tuple[str,str], np.ndarray]:
    """Load all files into memory. Key = (subject_id, label)."""
    cache: dict[tuple[str,str], np.ndarray] = {}
    skipped = 0
    for _, row in registry.iterrows():
        data = load_subject_epochs(Path(row["filepath"]))
        if data is not None:
            cache[(row["subject_id"], row["label"])] = data
        else:
            skipped += 1
    print(f"  Loaded {len(cache)} arrays | Skipped {skipped}")
    return cache


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — β-VAE
# ─────────────────────────────────────────────────────────────────────────────

class Sampling(tf.keras.layers.Layer):
    def call(self, inputs):
        mu, log_sigma = inputs
        eps = tf.random.normal(shape=tf.shape(mu), seed=SEED)
        return mu + tf.exp(0.5 * log_sigma) * eps


def build_encoder(
    n_ch: int = N_CHANNELS,
    n_t:  int = N_SAMPLES,
    ld:   int = LATENT_DIM,
) -> tf.keras.Model:
    """
    (batch, 19, 512) → [μ, log_σ, z]  each (batch, 64)

    DepthwiseConv1D(k=25) → PointwiseConv1D(64) → Permute
    → StrideConv1D×3(stride=2) → BiGRU(128) → Dense(128)
    → Dense μ, Dense log_σ, Sampling z
    """
    inp = tf.keras.Input(shape=(n_ch, n_t), name="enc_in")
    x   = tf.keras.layers.DepthwiseConv1D(
            kernel_size=25, padding="same",
            activation="elu", name="dw_conv")(inp)
    x   = tf.keras.layers.Conv1D(
            64, kernel_size=1,
            activation="elu", name="pw_conv")(x)
    x   = tf.keras.layers.Permute((2, 1), name="enc_perm")(x)
    for i, f in enumerate([64, 128, 128]):
        x = tf.keras.layers.Conv1D(
                f, kernel_size=5, strides=2,
                padding="same", activation="elu",
                name=f"sc_{i}")(x)
    x   = tf.keras.layers.Bidirectional(
            tf.keras.layers.GRU(128), name="bigru")(x)
    x   = tf.keras.layers.Dense(128, activation="elu", name="enc_d")(x)
    mu      = tf.keras.layers.Dense(ld, name="mu")(x)
    log_sig = tf.keras.layers.Dense(ld, name="log_sigma")(x)
    z       = Sampling(name="z")([mu, log_sig])
    return tf.keras.Model(inp, [mu, log_sig, z], name="encoder")


def build_decoder(
    n_ch: int = N_CHANNELS,
    n_t:  int = N_SAMPLES,
    ld:   int = LATENT_DIM,
) -> tf.keras.Model:
    """
    z (batch, 64) → (batch, 19, 512)

    Dense(64*128) → Reshape(64,128) → GRU(128)
    → ConvTranspose1D×3(stride=2) → Conv1D(19) → Permute
    512 = 64 * 2^3 → exact, no crop needed
    """
    z_in = tf.keras.Input(shape=(ld,), name="dec_in")
    x    = tf.keras.layers.Dense(
               64*128, activation="elu", name="dec_d")(z_in)
    x    = tf.keras.layers.Reshape((64, 128), name="reshape")(x)
    x    = tf.keras.layers.GRU(
               128, return_sequences=True, name="dec_gru")(x)
    for i, f in enumerate([128, 64, 64]):
        x = tf.keras.layers.Conv1DTranspose(
                f, kernel_size=5, strides=2,
                padding="same", activation="elu",
                name=f"dconv_{i}")(x)
    x    = tf.keras.layers.Conv1D(
               n_ch, kernel_size=1,
               activation="linear", name="out_conv")(x)
    x    = tf.keras.layers.Permute((2, 1), name="out_perm")(x)
    return tf.keras.Model(z_in, x, name="decoder")


class BetaVAE(tf.keras.Model):
    """
    β-VAE. Loss = MSE(x, x̂) + β·KL(q(z|x)‖N(0,I))
    Anomaly score = per-epoch MSE(x, x̂). Higher = more anomalous.
    """
    def __init__(self, encoder, decoder, beta=BETA):
        super().__init__()
        self.encoder     = encoder
        self.decoder     = decoder
        self.beta        = beta
        self._total      = tf.keras.metrics.Mean(name="total_loss")
        self._recon      = tf.keras.metrics.Mean(name="recon_loss")
        self._kl         = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self._total, self._recon, self._kl]

    def call(self, x, training=False):
        mu, log_sig, z = self.encoder(x, training=training)
        return self.decoder(z, training=training), mu, log_sig

    def _loss(self, x):
        x_hat, mu, log_sig = self(x, training=True)
        recon = tf.reduce_mean(
            tf.reduce_sum(tf.square(x - x_hat), axis=[1, 2]))
        kl    = -0.5 * tf.reduce_mean(
            tf.reduce_sum(
                1.0 + log_sig - tf.square(mu) - tf.exp(log_sig),
                axis=1))
        return recon + self.beta * kl, recon, kl

    def train_step(self, data):
        with tf.GradientTape() as tape:
            total, recon, kl = self._loss(data)
        self.optimizer.apply_gradients(
            zip(tape.gradient(total, self.trainable_variables),
                self.trainable_variables))
        self._total.update_state(total)
        self._recon.update_state(recon)
        self._kl.update_state(kl)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        total, recon, kl = self._loss(data)
        self._total.update_state(total)
        self._recon.update_state(recon)
        self._kl.update_state(kl)
        return {m.name: m.result() for m in self.metrics}

    def anomaly_score(self, x: np.ndarray) -> np.ndarray:
        """(N,19,512) → (N,) MSE scores. Higher = more epileptic."""
        xt        = tf.constant(x, dtype=tf.float32)
        x_hat,_,_ = self(xt, training=False)
        return tf.reduce_mean(
            tf.square(xt - x_hat), axis=[1, 2]).numpy().astype(np.float32)


# ── CHANGE 1: max_epochs parameter added ─────────────────────────────────────
def train_vae(
    X_train:    np.ndarray,
    max_epochs: int = EPOCHS_TRAIN,      # ← NEW: caller controls epoch budget
) -> BetaVAE:
    """
    Build and train a fresh VAE on X_train (normal epochs only).

    Parameters
    ----------
    X_train    : (N, 19, 512) normal epochs
    max_epochs : maximum training epochs before early stopping fires

    Returns
    -------
    Trained BetaVAE instance
    """
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    vae = BetaVAE(build_encoder(), build_decoder(), beta=BETA)
    vae.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE))

    ds = (tf.data.Dataset.from_tensor_slices(X_train)
          .shuffle(min(len(X_train), 5000), seed=SEED)
          .batch(BATCH_SIZE)
          .prefetch(tf.data.AUTOTUNE))

    vae.fit(
        ds,
        epochs=max_epochs,               # ← uses parameter
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor="total_loss", mode="min",
            patience=5, restore_best_weights=True, verbose=0
        )],
        verbose=0,
    )
    return vae
# ─────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — Baselines
# ─────────────────────────────────────────────────────────────────────────────

def pca_scores(
    X_train: np.ndarray,
    X_test:  np.ndarray,
    n_components: int = 50,
) -> np.ndarray:
    """
    PCA reconstruction error as anomaly score.

    Parameters
    ----------
    X_train : (N_train, 19*512) flattened normal epochs
    X_test  : (N_test,  19*512) flattened test epochs

    Returns
    -------
    scores : (N_test,) float32
    """
    nc  = min(n_components, X_train.shape[0]-1, X_train.shape[1])
    pca = PCA(n_components=nc, random_state=SEED)
    pca.fit(X_train)
    recon  = pca.inverse_transform(pca.transform(X_test))
    scores = np.mean((X_test - recon)**2, axis=1)
    return scores.astype(np.float32)


def isolation_forest_scores(
    X_train: np.ndarray,
    X_test:  np.ndarray,
) -> np.ndarray:
    """
    Isolation Forest anomaly scores.
    Returns negated decision_function so higher = more anomalous.
    """
    clf = IsolationForest(
        n_estimators=200,
        contamination=0.1,
        random_state=SEED,
        n_jobs=-1,
    )
    clf.fit(X_train)
    scores = -clf.decision_function(X_test)
    return scores.astype(np.float32)


def one_class_svm_scores(
    X_train: np.ndarray,
    X_test:  np.ndarray,
) -> np.ndarray:
    """
    One-Class SVM anomaly scores.
    Negated decision_function so higher = more anomalous.
    Uses RBF kernel. Scales features first.
    """
    scaler  = StandardScaler()
    Xtr_sc  = scaler.fit_transform(X_train)
    Xte_sc  = scaler.transform(X_test)
    clf     = OneClassSVM(kernel="rbf", nu=0.1, gamma="scale")
    clf.fit(Xtr_sc)
    scores  = -clf.decision_function(Xte_sc)
    return scores.astype(np.float32)


def extract_bandpower_features(data: np.ndarray) -> np.ndarray:
    """
    Extract mean band power per channel per epoch.

    Parameters
    ----------
    data : (N, 19, 512) normalised EEG

    Returns
    -------
    features : (N, 95) float32  — 5 bands × 19 channels
    """
    from scipy import signal as sp_signal

    bands = [(0.5,4),(4,8),(8,13),(13,30),(30,45)]
    freqs, psd = sp_signal.welch(data, fs=SFREQ, nperseg=256, axis=2)

    features = []
    for flo, fhi in bands:
        mask = (freqs >= flo) & (freqs <= fhi)
        bp   = psd[:, :, mask].mean(axis=2)
        features.append(bp)

    return np.concatenate(features, axis=1).astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — Metrics
# ─────────────────────────────────────────────────────────────────────────────

def sensitivity_at_specificity(
    y_true: np.ndarray,
    scores: np.ndarray,
    target: float = 0.95,
) -> float:
    fpr, tpr, _ = roc_curve(y_true, scores)
    specs       = 1.0 - fpr
    idx         = np.where(specs >= target)[0]
    if len(idx) == 0:
        return float("nan")
    best = idx[np.argmin(np.abs(specs[idx] - target))]
    return float(tpr[best])


def compute_all_metrics(
    y_true:     np.ndarray,
    scores:     np.ndarray,
    latency_ms: float = 0.0,
    n_params:   int   = 0,
) -> dict:
    """Full metric set. Threshold = median of scores."""
    if len(np.unique(y_true)) < 2:
        return {}
    auroc   = float(roc_auc_score(y_true, scores))
    thresh  = float(np.median(scores))
    y_pred  = (scores >= thresh).astype(int)
    f1      = float(f1_score(y_true, y_pred, zero_division=0))
    s95     = sensitivity_at_specificity(y_true, scores, 0.95)
    tn,fp,fn,tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    sens    = tp / max(tp+fn, 1)
    spec    = tn / max(tn+fp, 1)
    return {
        "auroc":          round(auroc, 4),
        "f1":             round(f1,    4),
        "sensitivity":    round(float(sens), 4),
        "specificity":    round(float(spec), 4),
        "sens_at_95spec": round(float(s95),  4),
        "latency_ms":     round(latency_ms,  4),
        "n_params":       n_params,
    }


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — Experiment 1: LOSO Cross-Subject
# ────────────────────────────────────────────────────────────────────────��────

def run_experiment1_loso(
    cache:    dict,
    registry: pd.DataFrame,
) -> pd.DataFrame:
    """
    LOSO cross-subject experiment.
    Trains VAE + all baselines. One row per fold per model.
    Checkpoints after every fold — safe to interrupt and resume.
    """
    loso_subjects = sorted(
        registry[registry["in_loso_pool"]]["subject_id"].unique()
    )
    print(f"\n  [EXP 1] LOSO | {len(loso_subjects)} subjects")

    # ── CHANGE 2: checkpoint resume ──────────────────────────────────────────
    checkpoint_path   = RESULTS_DIR / "exp1_loso_checkpoint.csv"
    all_rows          = []
    completed_folds   = set()

    if checkpoint_path.exists():
        existing        = pd.read_csv(checkpoint_path)
        all_rows        = existing.to_dict("records")
        completed_folds = set(existing["subject_id"].unique())
        print(f"  Resuming from checkpoint: "
              f"{len(completed_folds)} folds already done")
    # ─────────────────────────────────────────────────────────────────────────

    for fold_idx, test_sid in enumerate(loso_subjects):

        # ── CHANGE 2: skip completed folds ───────────────────────────────────
        if test_sid in completed_folds:
            print(f"  Fold {fold_idx+1:03d}/{len(loso_subjects)} "
                  f"| {test_sid} | SKIPPED (checkpoint)")
            continue
        # ─────────────────────────────────────────────────────────────────────

        # ── Training data: normal epochs, all subjects except test ───────────
        train_arrays = [
            data for (sid, lbl), data in cache.items()
            if sid != test_sid and lbl == "normal"
        ]
        if not train_arrays:
            continue

        X_train = np.concatenate(train_arrays, axis=0)
        np.random.shuffle(X_train)

        # ── Test data ───────────────���─────────────────────────────────────────
        X_n = cache.get((test_sid, "normal"))
        X_e = cache.get((test_sid, "epileptic"))
        if X_n is None or X_e is None:
            continue

        X_test = np.concatenate([X_n, X_e], axis=0)
        y_test = np.array([0]*len(X_n) + [1]*len(X_e), dtype=np.int32)

        print(f"  Fold {fold_idx+1:03d}/{len(loso_subjects)} | {test_sid} | "
              f"train_n={len(X_train)} "
              f"test_n={len(X_n)} test_e={len(X_e)}")

        # ── β-VAE — CHANGE 3: max_epochs=20 for Exp1 ─────────────────────────
        vae      = train_vae(X_train, max_epochs=20)
        # ─────────────────────────────────────────────────────────────────────
        n_params = int(sum(np.prod(v.shape) for v in vae.trainable_variables))
        _        = vae.anomaly_score(X_test[:1])
        t0       = time.perf_counter()
        vae_sc   = vae.anomaly_score(X_test)
        lat_vae  = ((time.perf_counter()-t0)/len(X_test))*1000

        m = compute_all_metrics(y_test, vae_sc, lat_vae, n_params)
        if m:
            all_rows.append({
                "experiment": "loso", "model": "beta_vae",
                "subject_id": test_sid, "fold": fold_idx+1, **m,
            })

        # ── PCA baseline ──────────────────────────────────────────────────────
        Xtr_flat = X_train.reshape(len(X_train), -1)
        Xte_flat = X_test.reshape(len(X_test),   -1)

        t0      = time.perf_counter()
        pca_sc  = pca_scores(Xtr_flat, Xte_flat)
        lat_pca = ((time.perf_counter()-t0)/len(X_test))*1000

        m = compute_all_metrics(y_test, pca_sc, lat_pca, 0)
        if m:
            all_rows.append({
                "experiment": "loso", "model": "pca",
                "subject_id": test_sid, "fold": fold_idx+1, **m,
            })

        # ── Isolation Forest (band-power features) ────────────────────────────
        Xtr_bp = extract_bandpower_features(X_train)
        Xte_bp = extract_bandpower_features(X_test)

        t0     = time.perf_counter()
        if_sc  = isolation_forest_scores(Xtr_bp, Xte_bp)
        lat_if = ((time.perf_counter()-t0)/len(X_test))*1000

        m = compute_all_metrics(y_test, if_sc, lat_if, 0)
        if m:
            all_rows.append({
                "experiment": "loso", "model": "isolation_forest",
                "subject_id": test_sid, "fold": fold_idx+1, **m,
            })

        # Print fold summary
        vae_auroc = next(
            r["auroc"] for r in reversed(all_rows)
            if r["model"]=="beta_vae" and r["subject_id"]==test_sid
        )
        pca_auroc = next(
            r["auroc"] for r in reversed(all_rows)
            if r["model"]=="pca" and r["subject_id"]==test_sid
        )
        print(f"    VAE={vae_auroc:.4f}  PCA={pca_auroc:.4f}")

        # ── CHANGE 2: save checkpoint after every completed fold ──────────────
        pd.DataFrame(all_rows).to_csv(checkpoint_path, index=False)
        # ─────────────────────────────────────────────────────────────────────

    return pd.DataFrame(all_rows)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — Experiment 2: Within-Subject
# ─────────────────────────────────────────────────────────────────────────────

def run_experiment2_within_subject(
    cache:    dict,
    registry: pd.DataFrame,
) -> pd.DataFrame:
    """
    Within-subject experiment.
    For each subject with both classes:
      Train on 80% of their OWN normal epochs
      Test  on 20% normal + ALL epileptic epochs
    Checkpoints after every subject — safe to interrupt and resume.
    """
    loso_subjects = sorted(
        registry[registry["in_loso_pool"]]["subject_id"].unique()
    )
    print(f"\n  [EXP 2] Within-subject | {len(loso_subjects)} subjects")

    # ── CHANGE 2: checkpoint resume ──────────────────────────────────────────
    checkpoint_path      = RESULTS_DIR / "exp2_within_subject_checkpoint.csv"
    all_rows             = []
    completed_subjects   = set()

    if checkpoint_path.exists():
        existing           = pd.read_csv(checkpoint_path)
        all_rows           = existing.to_dict("records")
        completed_subjects = set(existing["subject_id"].unique())
        print(f"  Resuming from checkpoint: "
              f"{len(completed_subjects)} subjects already done")
    # ─────────────────────────────────────────────────────────────────────────

    for sid in loso_subjects:

        # ── CHANGE 2: skip completed subjects ────────────────────────────────
        if sid in completed_subjects:
            print(f"  {sid:<20} | SKIPPED (checkpoint)")
            continue
        # ─────────────────────────────────────────────────────────────────────

        X_n = cache.get((sid, "normal"))
        X_e = cache.get((sid, "epileptic"))

        if X_n is None or X_e is None:
            continue
        if len(X_n) < 20:
            print(f"  [SKIP] {sid}: only {len(X_n)} normal epochs")
            continue

        rng     = np.random.RandomState(SEED)
        idx     = rng.permutation(len(X_n))
        n_train = int(len(X_n) * WITHIN_TRAIN_RATIO)
        X_train     = X_n[idx[:n_train]]
        X_norm_test = X_n[idx[n_train:]]

        X_test = np.concatenate([X_norm_test, X_e], axis=0)
        y_test = np.array(
            [0]*len(X_norm_test) + [1]*len(X_e), dtype=np.int32
        )

        print(f"  Subject {sid:<15} | "
              f"train_n={len(X_train)}  "
              f"test_n={len(X_norm_test)}  "
              f"test_e={len(X_e)}")

        # ── β-VAE — max_epochs=50 for Exp2 (within-subject needs more) ───────
        vae      = train_vae(X_train, max_epochs=EPOCHS_TRAIN)
        # ─────────────────────────────────────────────────────────────────────
        n_params = int(sum(np.prod(v.shape) for v in vae.trainable_variables))
        _        = vae.anomaly_score(X_test[:1])
        t0       = time.perf_counter()
        vae_sc   = vae.anomaly_score(X_test)
        lat_vae  = ((time.perf_counter()-t0)/len(X_test))*1000

        m = compute_all_metrics(y_test, vae_sc, lat_vae, n_params)
        if m:
            all_rows.append({
                "experiment": "within_subject", "model": "beta_vae",
                "subject_id": sid,
                "n_train_normal":   len(X_train),
                "n_test_normal":    len(X_norm_test),
                "n_test_epileptic": len(X_e),
                **m,
            })

        # ── PCA baseline ──────────────────────────────────────────────────────
        Xtr_flat = X_train.reshape(len(X_train), -1)
        Xte_flat = X_test.reshape(len(X_test),   -1)
        t0       = time.perf_counter()
        pca_sc   = pca_scores(Xtr_flat, Xte_flat)
        lat_pca  = ((time.perf_counter()-t0)/len(X_test))*1000

        m = compute_all_metrics(y_test, pca_sc, lat_pca, 0)
        if m:
            all_rows.append({
                "experiment": "within_subject", "model": "pca",
                "subject_id": sid,
                "n_train_normal":   len(X_train),
                "n_test_normal":    len(X_norm_test),
                "n_test_epileptic": len(X_e),
                **m,
            })

        # ── One-Class SVM (band-power) ────────────────────────────────────────
        Xtr_bp  = extract_bandpower_features(X_train)
        Xte_bp  = extract_bandpower_features(X_test)
        t0      = time.perf_counter()
        svm_sc  = one_class_svm_scores(Xtr_bp, Xte_bp)
        lat_svm = ((time.perf_counter()-t0)/len(X_test))*1000

        m = compute_all_metrics(y_test, svm_sc, lat_svm, 0)
        if m:
            all_rows.append({
                "experiment": "within_subject", "model": "oc_svm",
                "subject_id": sid,
                "n_train_normal":   len(X_train),
                "n_test_normal":    len(X_norm_test),
                "n_test_epileptic": len(X_e),
                **m,
            })

        # ── CHANGE 2: save checkpoint after every completed subject ───────────
        pd.DataFrame(all_rows).to_csv(checkpoint_path, index=False)
        # ─────────────────────────────────────────────────────────────────────

    return pd.DataFrame(all_rows)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — Final Report
# ─────────────────────────────────────────────────────────────────────────────

def print_final_report(
    exp1: pd.DataFrame,
    exp2: pd.DataFrame,
) -> None:
    print("\n" + "═"*68)
    print("  FINAL RESULTS")
    print("═"*68)

    for exp_name, df in [("EXPERIMENT 1 — LOSO Cross-Subject", exp1),
                          ("EXPERIMENT 2 — Within-Subject",     exp2)]:
        print(f"\n  {exp_name}")
        print("  " + "─"*60)

        if df.empty:
            print("  No results.")
            continue

        metrics = ["auroc","f1","sensitivity","specificity","sens_at_95spec"]
        models  = df["model"].unique()

        print(f"  {'Model':<20} " +
              "  ".join(f"{m:<16}" for m in metrics))
        print("  " + "─"*60)

        for mdl in models:
            sub     = df[df["model"]==mdl]
            row_str = f"  {mdl:<20}"
            for m in metrics:
                if m in sub.columns:
                    col = sub[m].dropna()
                    row_str += f"  {col.mean():.4f}±{col.std():.3f}  "
            print(row_str)

        if "beta_vae" in models and "pca" in models:
            vae_auroc = df[df["model"]=="beta_vae"]["auroc"].mean()
            pca_auroc = df[df["model"]=="pca"]["auroc"].mean()
            delta     = vae_auroc - pca_auroc
            marker    = "✔ VAE adds value" if delta > 0.01 else "✘ VAE = PCA"
            print(f"\n  VAE AUROC={vae_auroc:.4f}  "
                  f"PCA AUROC={pca_auroc:.4f}  "
                  f"Δ={delta:+.4f}  {marker}")

    print("\n" + "═"*68)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 — Entrypoint
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    print("═"*68)
    print("  β-VAE EPILEPSY PIPELINE v3 — TWO EXPERIMENTS + BASELINES")
    print(f"  Seed={SEED} | N_SAMPLES={N_SAMPLES} | N_CHANNELS={N_CHANNELS}")
    print("═"*68)

    if not DATASET_DIR.exists():
        print(f"  ✘ DATASET_DIR not found: {DATASET_DIR}")
        return

    registry = build_registry(DATASET_DIR)
    print(f"\n  Total subjects     : {registry['subject_id'].nunique()}")
    print(f"  LOSO eval subjects : "
          f"{registry[registry['in_loso_pool']]['subject_id'].nunique()}")

    print("\n  Pre-loading all epochs...")
    cache = preload_all(registry)

    exp1 = run_experiment1_loso(cache, registry)
    exp2 = run_experiment2_within_subject(cache, registry)

    if not exp1.empty:
        p = RESULTS_DIR / "exp1_loso.csv"
        exp1.to_csv(p, index=False)
        print(f"\n  ✔ Exp1 saved: {p}")

    if not exp2.empty:
        p = RESULTS_DIR / "exp2_within_subject.csv"
        exp2.to_csv(p, index=False)
        print(f"  ✔ Exp2 saved: {p}")

    print_final_report(exp1, exp2)


if __name__ == "__main__":
    main()

2026-03-13 19:16:31.174363: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773429391.367200      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773429391.422466      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773429391.931366      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773429391.931404      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773429391.931408      55 computation_placer.cc:177] computation placer alr

════════════════════════════════════════════════════════════════════
  β-VAE EPILEPSY PIPELINE v3 — TWO EXPERIMENTS + BASELINES
  Seed=42 | N_SAMPLES=512 | N_CHANNELS=19
════════════════════════════════════════════════════════════════════

  Total subjects     : 111
  LOSO eval subjects : 101

  Pre-loading all epochs...
  Loaded 212 arrays | Skipped 0

  [EXP 1] LOSO | 101 subjects
  Fold 001/101 | female_1 | train_n=24071 test_n=369 test_e=142


I0000 00:00:1773429443.996265      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773429444.002070      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1773429455.253183     124 cuda_dnn.cc:529] Loaded cuDNN version 91002


    VAE=1.0000  PCA=1.0000
  Fold 002/101 | female_10 | train_n=24221 test_n=219 test_e=183
    VAE=1.0000  PCA=1.0000
  Fold 003/101 | female_11 | train_n=24345 test_n=95 test_e=112
    VAE=0.2250  PCA=0.2038
  Fold 004/101 | female_12 | train_n=24391 test_n=49 test_e=164
    VAE=0.3006  PCA=0.2801
  Fold 005/101 | female_13 | train_n=24285 test_n=155 test_e=213
    VAE=0.3031  PCA=0.3052
  Fold 006/101 | female_14 | train_n=24132 test_n=308 test_e=373
    VAE=1.0000  PCA=1.0000
  Fold 007/101 | female_15 | train_n=24178 test_n=262 test_e=302
    VAE=0.8647  PCA=0.8344
  Fold 008/101 | female_16 | train_n=24269 test_n=171 test_e=294
    VAE=0.5420  PCA=0.4930
  Fold 009/101 | female_17 | train_n=24350 test_n=90 test_e=238
    VAE=0.3632  PCA=0.3560
  Fold 010/101 | female_18 | train_n=24128 test_n=312 test_e=193
    VAE=1.0000  PCA=1.0000
  Fold 011/101 | female_19 | train_n=24237 test_n=203 test_e=304
    VAE=0.7966  PCA=0.7881
  Fold 012/101 | female_2 | train_n=24045 test_n=395 tes